In [ ]:

#!/usr/bin/env python
# -*- coding: utf-8 -*-

import os, re, sys, csv, hashlib, shutil, datetime as dt

# --------- Dependências críticas ---------
missing = []
try:
    import win32com.client as win32
except Exception:
    win32 = None
    missing.append("pywin32")
try:
    import pandas as pd
except Exception:
    pd = None
    missing.append("pandas")
try:
    from PyPDF2 import PdfReader
except Exception:
    PdfReader = None
    missing.append("PyPDF2")
try:
    import openpyxl  # para pandas.ExcelWriter(engine="openpyxl")
except Exception:
    missing.append("openpyxl")

if missing:
    print("⚠️ Dependências ausentes:", ", ".join(missing))
    print("   Instale no Anaconda Prompt:")
    if "pywin32" in missing:
        print("   pip install pywin32")
    if "pandas" in missing:
        print("   pip install pandas")
    if "openpyxl" in missing:
        print("   pip install openpyxl")
    if "PyPDF2" in missing:
        print("   pip install PyPDF2")

# ======================================================================
# Parâmetros fixos do seu ambiente
# ======================================================================
BASE_PROJETOS = r"C:\Users\126815\OneDrive - paguemenos.com.br\Área de Trabalho\NFS\2025"

PROJETOS = [
    "ESPECIAL",
    "LAUDO DE RISCO",
    "LAUDO DE RISCO CD",
    "LAUDO DE RISCO MATRIZ",
    "LOJAS NOVAS",
    "MISSOES HIGIENICAS",
    "OUTROS",
    "PROJETO ESPECIAL",
    "RESIZE",
    "RETROFIT",
    "VIRADA DE BANDEIRA",
]

LOOKBACK_DAYS = 15
SUBPASTA_PATH = ""       # ex.: "Engenharia/NFs" (subpasta dentro da Inbox); vazio = inbox padrão
DIAGNOSTICO = True
LOG_IGNORADOS = False

# ======================================================================
# UI Terminal (menu de projeto + fornecedor digitado)
# ======================================================================
def select_projeto_menu(opcoes: list) -> str:
    print("\n📁 Selecione o PROJETO:")
    for i, nome in enumerate(opcoes, start=1):
        print(f" {i}. {nome}")
    print(" 0. Cancelar")
    while True:
        try:
            op = int(input("Digite o número da opção: ").strip())
            if op == 0:
                print("Operação cancelada.")
                return None
            if 1 <= op <= len(opcoes):
                projeto = opcoes[op - 1]
                print(f"✅ Selecionado: {projeto}")
                return projeto
        except ValueError:
            pass
        print("Opção inválida. Tente novamente.")

def ask_fornecedor_nome() -> str:
    nome = input("\nDigite o NOME exato do FORNECEDOR (ex.: LUMICENTER): ").strip()
    while not nome:
        print("Fornecedor vazio. Tente novamente.")
        nome = input("Digite o NOME do FORNECEDOR: ").strip()
    return nome

# ======================================================================
# Utils locais
# ======================================================================
def _ensure_dirs(path): os.makedirs(path, exist_ok=True)

def _sanitize_filename(name: str) -> str:
    name = (name or "").strip().replace("\n", " ").replace("\r", " ")
    name = re.sub(r'[<>:"/\\|?*]', "_", name)
    name = re.sub(r"\s{2,}", " ", name)
    return name

def _md5_file(path: str) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""): h.update(chunk)
    return h.hexdigest()

def _normalize_text(text: str) -> str:
    if not text: return ""
    t = re.sub(r"[\n\r\t]+", " ", text)
    t = re.sub(r"\s{2,}", " ", t)
    return t.upper()

def _parse_brl_to_float(v):
    if v is None: return None
    s = str(v)
    s = re.sub(r"[Rr]\$\s*", "", s)
    s = s.replace(".", "TEMP").replace(",", ".").replace("TEMP", "")
    s = re.sub(r"[^\d\.]", "", s)
    try: return float(s)
    except: return None

def _to_naive_local(d: dt.datetime) -> dt.datetime or None:
    if d is None: return None
    try:
        if getattr(d, "tzinfo", None) is not None:
            return d.astimezone().replace(tzinfo=None)
        return d
    except Exception:
        try: return dt.datetime.fromtimestamp(d.timestamp()).replace(tzinfo=None)
        except Exception: return None

# ======================================================================
# CNPJ & Loja
# ======================================================================
def _cnpj_digits(s: str) -> str:
    d = re.sub(r"\D", "", s or "")
    return d if len(d) == 14 else ""

def _cnpj_is_valid(d: str) -> bool:
    d = _cnpj_digits(d)
    if len(d) != 14 or len(set(d)) == 1: return False
    def dv_calc(digs, pesos):
        soma = sum(int(digs[i]) * pesos[i] for i in range(len(pesos)))
        r = soma % 11
        return '0' if r < 2 else str(11 - r)
    base = d[:12]
    dv1 = dv_calc(base, [5,4,3,2,9,8,7,6,5,4,3,2])
    dv2 = dv_calc(base + dv1, [6,5,4,3,2,9,8,7,6,5,4,3,2])
    return d[-2:] == dv1 + dv2

def _seq_from_cnpj_digits(cnpj_digits: str) -> str:
    if len(cnpj_digits) == 14: return cnpj_digits[8:12]
    return ""

def _derive_loja_from_cnpj(cnpj_raw: str, origem_cnpj: str) -> tuple:
    d = _cnpj_digits(cnpj_raw)
    seq = _seq_from_cnpj_digits(d) or (re.search(r"/(\d{4})-", cnpj_raw or "") or [None, ""])[1]
    origem_loja = origem_cnpj
    if not seq: return None, origem_loja
    if d.startswith("04"):
        loja = "7" + seq[1:]; origem_loja += " + Regra IMIFARMA"
    elif d.startswith("06"):
        loja = seq.lstrip("0") or "0"; origem_loja += " + Regra Pague Menos"
    else:
        loja = seq.lstrip("0") or seq
    return loja, origem_loja

# ======================================================================
# Padrões de CNPJ / heurística
# ======================================================================
CNPJ_FORMATADO = r"([0-9]{2}\.[0-9]{3}\.[0-9]{3}/[0-9]{4}\-[0-9]{2})"
CNPJ_GENERIC   = r"(\d{2}\D?\d{3}\D?\d{3}\D?\d{4}\D?\d{2})"
LABEL_CNPJ_NEAR = r"(?:CNPJ\s*/\s*CPF|CNPJ|CPF|C\.N\.P\.J\.)[^0-9]{0,200}" + CNPJ_FORMATADO

LABELS_DEST_PRIOR = [
    # DANFE usuais
    r"DESTINAT[ÁA]RIO\s*/\s*REMETENTE", r"DEST\.?\s*/\s*REM\.?",
    r"DESTINAT[ÁA]RIO", r"DADOS\s*DO\s*DESTINAT[ÁA]RIO",
    r"IDENTIFICA[ÇC][ÃA]O\s*DO\s*DESTINAT[ÁA]RIO",
    r"INFORMA[ÇC][ÕO]ES\s*DO\s*LOCAL\s*DE\s*ENTREGA",
    r"INFORMACOES\s*DO\s*LOCAL\s*DE\s*ENTREGA",
    # NFS-e
    r"DADOS\s*DO\s*TOMADOR\s*DE\s*SERVI[ÇC]OS",
    r"TOMADOR\s*DE\s*SERVI[ÇC]OS",
]

def _find_cnpj_candidates(text_upper: str) -> list:
    cands = []
    for m in re.finditer(LABEL_CNPJ_NEAR, text_upper, re.IGNORECASE):
        c_raw = m.group(1); d = _cnpj_digits(c_raw)
        if _cnpj_is_valid(d): cands.append((m.start(), c_raw, "CNPJ perto de rótulo"))
    for lbl in LABELS_DEST_PRIOR:
        m_lbl = re.search(lbl, text_upper)
        if m_lbl:
            start = max(0, m_lbl.start() - 4000)
            bloco = text_upper[start: m_lbl.start() + 12000]
            for m_fmt in re.finditer(CNPJ_FORMATADO, bloco):
                c_raw = m_fmt.group(1)
                if _cnpj_is_valid(_cnpj_digits(c_raw)):
                    cands.append((start + m_fmt.start(), c_raw, f"CNPJ perto de '{lbl}' (±)"))
            for m_gen in re.finditer(CNPJ_GENERIC, bloco):
                c_raw = m_gen.group(1)
                if _cnpj_is_valid(_cnpj_digits(c_raw)):
                    cands.append((start + m_gen.start(), c_raw, f"CNPJ genérico perto de '{lbl}' (±)"))
    for m_all in re.finditer(CNPJ_GENERIC, text_upper):
        c_raw = m_all.group(1); d = _cnpj_digits(c_raw)
        if _cnpj_is_valid(d): cands.append((m_all.start(), c_raw, "CNPJ por varredura"))
    seen, uniq = set(), []
    for pos, c_raw, src in cands:
        d = _cnpj_digits(c_raw)
        if d not in seen:
            seen.add(d); uniq.append((pos, c_raw, src))
    return uniq

def _pick_best_dest_cnpj(text_upper: str, candidates: list) -> tuple:
    if not candidates: return None, None
    pos_hints  = ("DESTINAT", "REMETENTE", "LOCAL DE ENTREGA", "TOMADOR")
    house_hints = ("PAGUE MENOS", "EMPREEN")
    neg_hints  = ("TRANSPORTADOR", "FRETE POR CONTA", "EMITENTE", "PRESTADOR", "PROTOCOLO", "CHAVE DE ACESSO")
    best = None; best_score = -10**9; best_src = None
    for pos, c_raw, src in candidates:
        d   = _cnpj_digits(c_raw)
        win = text_upper[max(0, pos-2000): pos+2000]
        s = 0
        if d.startswith("04"): s += 3
        if d.startswith("06"): s += 2
        if any(lbl in win for lbl in pos_hints):  s += 6
        if any(lbl in win for lbl in house_hints): s += 4
        if any(lbl in win for lbl in neg_hints):  s -= 5
        if s > best_score:
            best_score = s; best = c_raw; best_src = f"{src} (score={s})"
    return best, best_src

# ======================================================================
# Extratores
# ======================================================================
def extrair_vencimento_de_texto(texto: str) -> str or None:
    if not texto: return None
    t = _normalize_text(texto).upper()
    m = re.search(r"\b(VENC(?:IMENTO)?|VENCTO|VCTO)\b[^0-9]{0,10}(\d{2}/\d{2}/\d{4})", t)
    if m: return m.group(2)
    m2 = re.search(r"\b(\d{2}/\d{2}/\d{4})\b", t)
    return m2.group(1) if m2 else None

def _find_oc_candidates(text_upper: str) -> list:
    oc_pattern = r"\b(45\d{8,10})\b"
    cands = []
    labels = [
        "ORDEM DE COMPRA", "OC", "PEDIDO", "PEDIDO CLIENTE", "PEDIDO DE VENDAS",
        "DADOS ADICIONAIS", "INFORMAÇÕES COMPLEMENTARES", "DISCRIMINAÇÃO DOS SERVIÇOS",
    ]
    for lbl in labels:
        m_lbl = re.search(lbl, text_upper)
        if m_lbl:
            start = max(0, m_lbl.start()-2000)
            bloco = text_upper[start: m_lbl.start()+4000]
            for m in re.finditer(oc_pattern, bloco):
                cands.append((start + m.start(), m.group(1), 5))
    for m in re.finditer(oc_pattern, text_upper):
        cands.append((m.start(), m.group(1), 1))
    seen, uniq = set(), []
    for pos, oc, score in cands:
        if oc not in seen:
            seen.add(oc); uniq.append((pos, oc, score))
    return uniq

def extrair_dados_pdf(caminho_pdf, debug=False):
    if PdfReader is None:
        return None, None, None, None, None, None, {}
    try:
        reader = PdfReader(caminho_pdf)
        texto_completo = ""
        for page in reader.pages:
            texto_completo += (page.extract_text() or "") + "\n"
        t_upper = _normalize_text(texto_completo)
        is_nfse = "NFS-E" in t_upper or "SERVIÇO" in t_upper or "PREFEITURA" in t_upper

        cands = _find_cnpj_candidates(t_upper)
        cnpj_dest_raw, src_dest = _pick_best_dest_cnpj(t_upper, cands)
        cnpj_dest = _cnpj_digits(cnpj_dest_raw or "")
        loja, origem_loja = _derive_loja_from_cnpj(cnpj_dest, src_dest or "não encontrado")

        if not loja:
            loja_bloco = None
            if cnpj_dest_raw:
                pos = t_upper.find(cnpj_dest_raw)
                loja_bloco = t_upper[max(0, pos-1500): pos+2500]
            if not loja_bloco:
                m_dest = re.search(r"(DESTINAT[ÁA]RIO\s*/\s*REMETENTE.{0,2000})", t_upper)
                loja_bloco = m_dest.group(1) if m_dest else t_upper
            m_lj = re.search(r"(?:LOJA|FILIAL|PAGUE\s*MENOS)[^\d]{0,15}(\d{1,5})", loja_bloco)
            if m_lj:
                loja = m_lj.group(1); origem_loja = "regex bloco DESTINATÁRIO/REMETENTE"

        nf, serie = None, None
        if is_nfse:
            m_belem = re.search(r"N[ÚU]MERO\s*/\s*S[ÉE]RIE\s*(\d+)\s*/\s*([A-Z0-9]+)", t_upper)
            if m_belem: nf, serie = m_belem.group(1), m_belem.group(2)
        if not nf:
            m_nf = re.search(r"(?:N[ÚU]MERO|NF|NFS-?E)[^0-9]{0,50}(\d{3,})", t_upper)
            if m_nf: nf = m_nf.group(1)
        if not serie:
            m_s = re.search(r"S[ÉE]RIE[^0-9A-Z]{0,50}(\d{1,3}|[A-Z])", t_upper)
            if m_s: serie = m_s.group(1)

        valor_total_str = None
        if is_nfse:
            all_vals = re.findall(r"([\d\.]{1,12},\d{2})", t_upper)
            numeric_vals = []
            for v in all_vals:
                try: numeric_vals.append(float(v.replace(".", "").replace(",", ".")))
                except: pass
            if numeric_vals:
                max_val = max(numeric_vals)
                valor_total_str = f"{max_val:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
        if not valor_total_str:
            m_v = re.search(
                r"(VALOR\s*(?:TOTAL|L[ÍI]QUIDO|DA\s*NOTA|DOS\s*SERVI[ÇC]OS)|TOTAL\s*(?:A\s*PAGAR|GERAL))[^0-9]{0,40}([\d\.,]{2,})",
                t_upper
            )
            if m_v: valor_total_str = m_v.group(2)
        if not valor_total_str:
            m_v2 = re.search(r"TOTAL[^0-9]{0,25}([\d\.,]{4,})", t_upper)
            if m_v2: valor_total_str = m_v2.group(1)

        chave = (re.search(r"(?:CHAVE\s*DE\s*ACESSO|CHAVE)[^0-9]{0,60}(\d{44})", t_upper) or [None, None])[1]
        cfop  = (re.search(r"CFOP[^0-9]{0,60}(\d{4})", t_upper) or [None, None])[1]

        emit_cnpj = None; emit_nome = None
        for pos, c_raw, src in cands:
            d = _cnpj_digits(c_raw)
            if d and d != cnpj_dest:
                win = t_upper[max(0, pos-800): pos+800]
                if any(x in win for x in ["EMITENTE", "PRESTADOR", "NOME / NOME EMPRESARIAL", "IDENTIFICAÇÃO DO EMITENTE"]):
                    emit_cnpj = d
                    m_nome = re.search(r"(?:NOME\s*/\s*NOME\s*EMPRESARIAL|IDENTIFICAÇÃO DO EMITENTE)\s*([^0-9\n]{3,120})", win)
                    if m_nome: emit_nome = m_nome.group(1).strip()
                    break
        if not emit_cnpj and cands:
            for pos, c_raw, src in cands:
                d = _cnpj_digits(c_raw)
                if d and d != cnpj_dest:
                    emit_cnpj = d; break

        oc_cands = _find_oc_candidates(t_upper)
        oc = max(oc_cands, key=lambda x: x[2])[1] if oc_cands else None
        data_emissao = (re.search(r"(?:DATA\s*DE\s*EMISS[ÃA]O|EMISS[ÃA]O|DATA\s*E\s*HORA\s*DE\s*EMISS[ÃA]O)[^0-9]{0,60}(\d{2}/\d{2}/\d{4})", t_upper) or [None, None])[1]

        extras = {
            "Valor Total": valor_total_str,
            "ordem_compra": oc,
            "pedido": oc,
            "cfop": cfop,
            "emitente_cnpj": emit_cnpj,
            "emitente_nome": emit_nome,
            "destinatario_cnpj": cnpj_dest,
            "data_emissao": data_emissao,
            "data_vencimento": None,
        }
        return nf, loja, "extração PDF", origem_loja, serie, chave, extras

    except Exception as e:
        print(f"⚠️ Erro ao ler {os.path.basename(caminho_pdf)}: {e}")
        return None, None, None, None, None, None, {}

# ======================================================================
# Outlook multi-Inbox (Win32)
# ======================================================================
PR_SMTP_ADDRESS = "http://schemas.microsoft.com/mapi/proptag/0x39FE001E"

SUBJECT_PATTERNS = [
    r"\bnota\s*fiscal\b",
    r"\bnf-?e\b",
    r"\bdanfe\b",
    r"\bnf\b",
    r"\bnfs-?e\b",
    r"\bservi[cç]o\b",
]

def subject_is_nf(subject: str) -> bool:
    s = subject or ""
    for p in SUBJECT_PATTERNS:
        if re.search(p, s, re.IGNORECASE): return True
    return False

def _canon_email(e: str) -> str:
    e = (e or "").strip().lower()
    return e[5:] if e.startswith("smtp:") else e

def resolve_sender_smtp(mail_item):
    smtp = ""
    try:
        sender = getattr(mail_item, "Sender", None)
        if sender:
            ae = sender.AddressEntry
            if ae:
                pa = ae.PropertyAccessor
                try:
                    smtp = (pa.GetProperty(PR_SMTP_ADDRESS) or "").lower()
                except:
                    smtp = ""
        if not smtp:
            smtp = (getattr(mail_item, "SenderEmailAddress", "") or "").lower()
            if smtp.startswith("smtp:"): smtp = smtp[5:]
    except:
        smtp = (getattr(mail_item, "SenderEmailAddress", "") or "").lower()
    return smtp

def _match_sender_or_subject(sender_addr: str, allow_list: list, sender_name: str, subject: str) -> (bool, str):
    if not allow_list:
        if subject_is_nf(subject): return True, "sem filtro, match por assunto NF"
        return False, "sem filtro, assunto não indica NF"
    s = _canon_email(sender_addr)
    if s in allow_list: return True, "match exato smtp"
    if any(s.endswith(x) for x in allow_list): return True, "match por sufixo smtp"
    return False, f"sender '{s}' não está na lista"

def resolve_subfolder(root_folder, path_str):
    if not path_str: return root_folder
    cur = root_folder
    for part in path_str.split("/"):
        if not part: continue
        cur = cur.Folders(part)
    return cur

def get_all_candidate_inboxes(namespace, subpath=""):
    candidates = []
    try:
        stores = namespace.Stores
        for i in range(1, stores.Count + 1):
            st = stores.Item(i)
            try:
                inbox = st.GetDefaultFolder(6)  # Inbox
                target = resolve_subfolder(inbox, subpath)
                candidates.append(("STORE", st.DisplayName, target))
            except: pass
    except: pass
    try:
        for i in range(1, namespace.Session.Accounts.Count + 1):
            acc = namespace.Session.Accounts.Item(i)
            try:
                ds = acc.DeliveryStore
                inbox = ds.GetDefaultFolder(6)
                target = resolve_subfolder(inbox, subpath)
                candidates.append(("ACCOUNT", acc.SmtpAddress, target))
            except: pass
    except: pass
    try:
        inbox = namespace.GetDefaultFolder(6)
        target = resolve_subfolder(inbox, subpath)
        candidates.append(("DEFAULT", "DefaultStore", target))
    except: pass

    seen, uniq = set(), []
    for kind, name, folder in candidates:
        try: fp = folder.FolderPath
        except: fp = f"{kind}:{name}"
        if fp not in seen:
            seen.add(fp); uniq.append((kind, name, folder))
    return uniq

def coletar_anexos_outlook_multi(filter_senders_lower):
    if win32 is None:
        print("❌ pywin32 não disponível ou Outlook não é o Clássico (Win32).")
        return []
    outlook = win32.Dispatch("Outlook.Application").GetNamespace("MAPI")

    dt_from_naive = (dt.datetime.now().replace(tzinfo=None) - dt.timedelta(days=LOOKBACK_DAYS))
    inboxes = get_all_candidate_inboxes(outlook, SUBPASTA_PATH.replace("\\", "/"))

    print("\n🧭 Inboxes candidatas localizadas:")
    for kind, name, folder in inboxes:
        try: print(f"   - {kind}: {name} | {folder.FolderPath}")
        except: print(f"   - {kind}: {name}")

    coletados, ignorados = [], []

    for kind, name, folder in inboxes:
        try:
            items = folder.Items
            items.Sort("[ReceivedTime]", True)  # DESC
            it = items.GetFirst()
            i_debug = 0
            while it:
                try:
                    received = getattr(it, "ReceivedTime", None)
                    received_naive = _to_naive_local(received)
                    if received_naive and received_naive < dt_from_naive: break
                    if not getattr(it, "Attachments", None):
                        it = items.GetNext(); continue

                    sender_name = (getattr(it, "SenderName", "") or "").strip()
                    sender_addr = resolve_sender_smtp(it)
                    subject = (getattr(it, "Subject", "") or "")
                    body = getattr(it, "Body", "")

                    match, motivo = _match_sender_or_subject(sender_addr, filter_senders_lower, sender_name, subject)
                    found_pdf = any(str(att.FileName or "").lower().endswith(".pdf") for att in it.Attachments)

                    if not match or not found_pdf:
                        ignorados.append({
                            "Inbox": getattr(folder, "FolderPath", f"{kind}:{name}"),
                            "Sender": sender_addr,
                            "SenderName": sender_name,
                            "Subject": subject[:200],
                            "Received": received_naive,
                            "TemPDF": found_pdf,
                            "Motivo": motivo if not match else "sem PDF"
                        })
                        it = items.GetNext(); continue

                    # Salvar PDFs em _temp local; depois movemos para a pasta do projeto/fornecedor
                    temp_dir = os.path.join(os.getcwd(), "_temp_nfs"); _ensure_dirs(temp_dir)
                    for att in it.Attachments:
                        fname = str(att.FileName or "")
                        if not fname.lower().endswith(".pdf"): continue
                        temp_path = os.path.join(temp_dir, _sanitize_filename(fname))
                        att.SaveAsFile(temp_path)
                        coletados.append({
                            "entry_id": it.EntryID,
                            "attach_id": f"{it.EntryID}::{fname}",
                            "sender_email": sender_addr,
                            "sender_name": sender_name,
                            "subject": subject,
                            "received": received_naive,
                            "body": body,
                            "temp_path": temp_path,
                            "orig_filename": fname,
                            "inbox_path": getattr(folder, "FolderPath", f"{kind}:{name}")
                        })

                    if DIAGNOSTICO and i_debug < 5:
                        print(f"   • {folder.FolderPath} | {received_naive} | From={sender_addr} | PDF=SIM | Subj={subject[:80]}")
                        i_debug += 1

                except Exception as e:
                    print(f"⚠️ Erro ao ler item ({getattr(folder,'FolderPath','')}): {e}")
                it = items.GetNext()
        except Exception as e:
            print(f"⚠️ Erro ao acessar pasta {getattr(folder,'FolderPath','')}: {e}")

    if DIAGNOSTICO and LOG_IGNORADOS and ignorados:
        print("\n📝 E-mails ignorados:")
        for ig in ignorados[:30]:
            print(f"   - {ig['Inbox']} | {ig['Received']} | From={ig['Sender']} | PDF={ig['TemPDF']} | Subj={ig['Subject']} | Motivo={ig['Motivo']}")
    return coletados

# ======================================================================
# MAIN
# ======================================================================
def main():
    # 1) Selecionar PROJETO
    projeto = select_projeto_menu(PROJETOS)
    if not projeto:
        return

    # 2) Digitar FORNECEDOR
    fornecedor = ask_fornecedor_nome()

    # 3) Montar caminhos
    pasta_projeto = os.path.join(BASE_PROJETOS, projeto)
    pasta_fornecedor = os.path.join(pasta_projeto, fornecedor)

    if not os.path.isdir(pasta_projeto):
        print(f"\n❌ A pasta do PROJETO não existe:\n   {pasta_projeto}\nCrie-a e rode novamente.")
        return

    if not os.path.isdir(pasta_fornecedor):
        print(f"\n❌ A pasta do FORNECEDOR não existe dentro do projeto:\n   {pasta_fornecedor}\n"
              f"Como solicitado, o script **NÃO** cria pastas automaticamente.")
        seguir = input("Deseja seguir apenas para gerar o Excel (sem mover PDFs)? [s/N]: ").strip().lower()
        if seguir not in ("s", "sim", "y", "yes"):
            return

    # 4) Informar remetentes (SMTP) a filtrar (ou vazio para TODOS com assunto NF/NFS-e/DANFE)
    raw_emails = input("\nDigite os e-mails de remetentes a filtrar (separados por vírgula; deixe vazio para TODOS): ").strip()
    FILTER_SENDERS = [e.strip().lower() for e in raw_emails.split(",") if e.strip()]
    if FILTER_SENDERS:
        print("✅ Filtrando por remetentes:", FILTER_SENDERS)
    else:
        print("ℹ️ Sem filtro de remetente: match por assunto (NF/NFS-e/DANFE).")

    # 5) Coletar anexos
    registros = coletar_anexos_outlook_multi(FILTER_SENDERS)
    print(f"\n📥 PDFs encontrados: {len(registros)}")

    # 6) Processar PDFs (mover somente se pasta do fornecedor existir)
    base_registros = []
    movidos = 0

    for reg in registros:
        pdf_path = reg["temp_path"]
        nf, loja, origem_nf, origem_loja, serie, chave, extras = extrair_dados_pdf(pdf_path, debug=False)

        if nf and loja:
            novo_nome = f"NF {nf} - LJ {loja}.pdf"
        elif nf:
            novo_nome = f"NF {nf}.pdf"
        else:
            base_nome = os.path.splitext(reg["orig_filename"])[0]
            novo_nome = _sanitize_filename(f"{base_nome} - PROCESSADO.pdf")

        destino_final = os.path.join(pasta_fornecedor, novo_nome)

        status = "SEM PASTA FORNECEDOR"
        if os.path.isdir(pasta_fornecedor):
            try:
                if os.path.exists(destino_final):
                    h = _md5_file(pdf_path)[:8]
                    destino_final = os.path.join(pasta_fornecedor, f"{os.path.splitext(novo_nome)[0]} - {h}.pdf")
                shutil.move(pdf_path, destino_final)
                status = "OK"
                movidos += 1
                print(f"✅ {os.path.basename(destino_final)}  ←  {reg.get('inbox_path','')}")
            except Exception as e:
                print(f"❌ Falha ao mover {pdf_path} -> {destino_final}: {e}")
                status = f"ERRO: {e}"
        else:
            print(f"⚠️ Pasta do fornecedor ausente, não movido: {novo_nome}")

        data_venc = extras.get("data_vencimento") or \
                    (subject_is_nf(reg["subject"]) and extrair_vencimento_de_texto(reg["subject"])) or \
                    extrair_vencimento_de_texto(reg["body"])

        base_registros.append({
            "Projeto": projeto,
            "Fornecedor": fornecedor,
            "Arquivo": os.path.basename(destino_final) if os.path.isdir(pasta_fornecedor) else novo_nome,
            "Caminho": destino_final if os.path.isdir(pasta_fornecedor) else "(não movido - sem pasta fornecedor)",
            "NF": nf,
            "Loja": loja,
            "Série": serie,
            "Chave": chave,
            "Valor Total": extras.get("Valor Total"),
            "Pedido": extras.get("pedido"),
            "Ordem de Compra": extras.get("ordem_compra"),
            "CFOP": extras.get("cfop"),
            "Emitente CNPJ": extras.get("emitente_cnpj"),
            "Destinatário CNPJ": extras.get("destinatario_cnpj"),
            "Data Emissão": extras.get("data_emissao"),
            "Vencimento": data_venc,
            "Assunto Email": reg["subject"],
            "Remetente": reg["sender_email"],
            "Remetente Nome": reg["sender_name"],
            "Recebido Em": reg["received"],
            "Inbox": reg.get("inbox_path", ""),
            "Status": status
        })

    # 7) Snapshot (execução atual) -> salva na pasta do PROJETO
    if base_registros and pd is not None:
        df_run = pd.DataFrame(base_registros)
        if "Recebido Em" in df_run.columns:
            df_run = df_run.sort_values(by="Recebido Em", ascending=False, ignore_index=True)

        # ======== AJUSTE CRÍTICO: agregação com rótulo 'Valor_Total_R$' ========
        work = df_run.copy()
        work["ValorNum"] = work["Valor Total"].apply(_parse_brl_to_float)

        # Usamos named aggregation com **{...} para permitir 'R$' no nome
        resumo = work.groupby(["Fornecedor","Status"], dropna=False).agg(
            Qtde_NFs=("NF", "count"),
            Qtde_com_Loja=("Loja", lambda s: s.notna().sum()),
            **{"Valor_Total_R$": ("ValorNum", "sum")},
            Primeiro_Recebido=("Recebido Em", "min"),
            Ultimo_Recebido=("Recebido Em", "max"),
        ).reset_index()

        resumo["Qtde_sem_Loja"] = resumo["Qtde_NFs"] - resumo["Qtde_com_Loja"]
        resumo = resumo.sort_values(["Status","Fornecedor"], ignore_index=True)

        def fmt_brl(x):
            try: return f"R$ {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
            except: return "R$ 0,00"

        print("\n📈 RESUMO (execução atual):\n")
        rprint = resumo.copy()
        rprint["Valor_Total_R$"] = rprint["Valor_Total_R$"].map(fmt_brl)
        print(rprint.to_string(index=False))

        hoje = dt.datetime.now().strftime("%Y-%m-%d")
        snap_path = os.path.join(pasta_projeto, f"BASE_NFs_{fornecedor}_{hoje}.xlsx")
        with pd.ExcelWriter(snap_path, engine="openpyxl") as writer:
            df_run.to_excel(writer, index=False, sheet_name="BASE")
            resumo.to_excel(writer, index=False, sheet_name="RESUMO")
        print(f"\n🗂️ Snapshot salvo em: {snap_path}")

    # 8) Limpa _temp se vazio
    temp_dir = os.path.join(os.getcwd(), "_temp_nfs")
    if os.path.isdir(temp_dir) and not os.listdir(temp_dir):
        try: os.rmdir(temp_dir)
        except: pass

    print(f"\n✨ Concluído. PDFs movidos: {movidos}")

# ---- RUN ----
if __name__ == "__main__":
    if not os.path.isdir(BASE_PROJETOS):
        print(f"❌ Caminho BASE dos projetos não existe:\n   {BASE_PROJETOS}")
    elif win32 is None:
        print("\n❌ pywin32 não disponível ou Outlook não é o Clássico (Win32).")
        print("   Garanta: Outlook Clássico aberto/logado + 'pip install pywin32 pandas openpyxl PyPDF2'\n")
    else:
        main()



📁 Selecione o PROJETO:
 1. ESPECIAL
 2. LAUDO DE RISCO
 3. LAUDO DE RISCO CD
 4. LAUDO DE RISCO MATRIZ
 5. LOJAS NOVAS
 6. MISSOES HIGIENICAS
 7. OUTROS
 8. PROJETO ESPECIAL
 9. RESIZE
 10. RETROFIT
 11. VIRADA DE BANDEIRA
 0. Cancelar


In [1]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

import os, re, sys, csv, hashlib, shutil, datetime as dt

# --------- Dependências críticas ---------
missing = []
try:
    import win32com.client as win32
except Exception:
    win32 = None
    missing.append("pywin32")
try:
    import pandas as pd
except Exception:
    pd = None
    missing.append("pandas")
try:
    from PyPDF2 import PdfReader
except Exception:
    PdfReader = None
    missing.append("PyPDF2")
try:
    import openpyxl  # para pandas.ExcelWriter(engine="openpyxl")
except Exception:
    missing.append("openpyxl")

if missing:
    print("⚠️ Dependências ausentes:", ", ".join(missing))
    print("   Instale no Anaconda Prompt:")
    if "pywin32" in missing:
        print("   pip install pywin32")
    if "pandas" in missing:
        print("   pip install pandas")
    if "openpyxl" in missing:
        print("   pip install openpyxl")
    if "PyPDF2" in missing:
        print("   pip install PyPDF2")

# ======================================================================
# Parâmetros fixos do seu ambiente
# ======================================================================
BASE_PROJETOS = r"C:\Users\126815\OneDrive - paguemenos.com.br\Área de Trabalho\NFS\2025"

PROJETOS = [
    "ESPECIAL",
    "LAUDO DE RISCO",
    "LAUDO DE RISCO CD",
    "LAUDO DE RISCO MATRIZ",
    "LOJAS NOVAS",
    "MISSOES HIGIENICAS",
    "OUTROS",
    "PROJETO ESPECIAL",
    "RESIZE",
    "RETROFIT",
    "VIRADA DE BANDEIRA",
]

LOOKBACK_DAYS = 15
SUBPASTA_PATH = ""       # ex.: "Engenharia/NFs" (subpasta dentro da Inbox); vazio = inbox padrão
DIAGNOSTICO = True
LOG_IGNORADOS = False

# ======================================================================
# UI Terminal (menu de projeto + fornecedor digitado)
# ======================================================================
def select_projeto_menu(opcoes: list) -> str:
    print("\n📁 Selecione o PROJETO:")
    for i, nome in enumerate(opcoes, start=1):
        print(f" {i}. {nome}")
    print(" 0. Cancelar")
    while True:
        try:
            op = int(input("Digite o número da opção: ").strip())
            if op == 0:
                print("Operação cancelada.")
                return None
            if 1 <= op <= len(opcoes):
                projeto = opcoes[op - 1]
                print(f"✅ Selecionado: {projeto}")
                return projeto
        except ValueError:
            pass
        print("Opção inválida. Tente novamente.")

def ask_fornecedor_nome() -> str:
    nome = input("\nDigite o NOME exato do FORNECEDOR (ex.: LUMICENTER): ").strip()
    while not nome:
        print("Fornecedor vazio. Tente novamente.")
        nome = input("Digite o NOME do FORNECEDOR: ").strip()
    return nome

# ======================================================================
# Utils locais
# ======================================================================
def _ensure_dirs(path): os.makedirs(path, exist_ok=True)

def _sanitize_filename(name: str) -> str:
    name = (name or "").strip().replace("\n", " ").replace("\r", " ")
    name = re.sub(r'[<>:"/\\|?*]', "_", name)
    name = re.sub(r"\s{2,}", " ", name)
    return name

def _md5_file(path: str) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""): h.update(chunk)
    return h.hexdigest()

def _normalize_text(text: str) -> str:
    if not text: return ""
    t = re.sub(r"[\n\r\t]+", " ", text)
    t = re.sub(r"\s{2,}", " ", t)
    return t.upper()

def _parse_brl_to_float(v):
    if v is None: return None
    s = str(v)
    s = re.sub(r"[Rr]\$\s*", "", s)
    s = s.replace(".", "TEMP").replace(",", ".").replace("TEMP", "")
    s = re.sub(r"[^\d\.]", "", s)
    try: return float(s)
    except: return None

def _to_naive_local(d: dt.datetime) -> dt.datetime or None:
    if d is None: return None
    try:
        if getattr(d, "tzinfo", None) is not None:
            return d.astimezone().replace(tzinfo=None)
        return d
    except Exception:
        try: return dt.datetime.fromtimestamp(d.timestamp()).replace(tzinfo=None)
        except Exception: return None

# ======================================================================
# CNPJ & Loja
# ======================================================================
def _cnpj_digits(s: str) -> str:
    d = re.sub(r"\D", "", s or "")
    return d if len(d) == 14 else ""

def _cnpj_is_valid(d: str) -> bool:
    d = _cnpj_digits(d)
    if len(d) != 14 or len(set(d)) == 1: return False
    def dv_calc(digs, pesos):
        soma = sum(int(digs[i]) * pesos[i] for i in range(len(pesos)))
        r = soma % 11
        return '0' if r < 2 else str(11 - r)
    base = d[:12]
    dv1 = dv_calc(base, [5,4,3,2,9,8,7,6,5,4,3,2])
    dv2 = dv_calc(base + dv1, [6,5,4,3,2,9,8,7,6,5,4,3,2])
    return d[-2:] == dv1 + dv2

def _seq_from_cnpj_digits(cnpj_digits: str) -> str:
    if len(cnpj_digits) == 14: return cnpj_digits[8:12]
    return ""

def _derive_loja_from_cnpj(cnpj_raw: str, origem_cnpj: str) -> tuple:
    d = _cnpj_digits(cnpj_raw)
    seq = _seq_from_cnpj_digits(d) or (re.search(r"/(\d{4})-", cnpj_raw or "") or [None, ""])[1]
    origem_loja = origem_cnpj
    if not seq: return None, origem_loja
    if d.startswith("04"):
        loja = "7" + seq[1:]; origem_loja += " + Regra IMIFARMA"
    elif d.startswith("06"):
        loja = seq.lstrip("0") or "0"; origem_loja += " + Regra Pague Menos"
    else:
        loja = seq.lstrip("0") or seq
    return loja, origem_loja

# ======================================================================
# Padrões de CNPJ / heurística
# ======================================================================
CNPJ_FORMATADO = r"([0-9]{2}\.[0-9]{3}\.[0-9]{3}/[0-9]{4}\-[0-9]{2})"
CNPJ_GENERIC   = r"(\d{2}\D?\d{3}\D?\d{3}\D?\d{4}\D?\d{2})"
LABEL_CNPJ_NEAR = r"(?:CNPJ\s*/\s*CPF|CNPJ|CPF|C\.N\.P\.J\.)[^0-9]{0,200}" + CNPJ_FORMATADO

LABELS_DEST_PRIOR = [
    # DANFE usuais
    r"DESTINAT[ÁA]RIO\s*/\s*REMETENTE", r"DEST\.?\s*/\s*REM\.?",
    r"DESTINAT[ÁA]RIO", r"DADOS\s*DO\s*DESTINAT[ÁA]RIO",
    r"IDENTIFICA[ÇC][ÃA]O\s*DO\s*DESTINAT[ÁA]RIO",
    r"INFORMA[ÇC][ÕO]ES\s*DO\s*LOCAL\s*DE\s*ENTREGA",
    r"INFORMACOES\s*DO\s*LOCAL\s*DE\s*ENTREGA",
    # NFS-e
    r"DADOS\s*DO\s*TOMADOR\s*DE\s*SERVI[ÇC]OS",
    r"TOMADOR\s*DE\s*SERVI[ÇC]OS",
    r"TOMADOR\s*DO\s*SERVI[ÇC]O",
]

def _find_cnpj_candidates(text_upper: str) -> list:
    cands = []
    for m in re.finditer(LABEL_CNPJ_NEAR, text_upper, re.IGNORECASE):
        c_raw = m.group(1); d = _cnpj_digits(c_raw)
        if _cnpj_is_valid(d): cands.append((m.start(), c_raw, "CNPJ perto de rótulo"))
    for lbl in LABELS_DEST_PRIOR:
        m_lbl = re.search(lbl, text_upper)
        if m_lbl:
            start = max(0, m_lbl.start() - 4000)
            bloco = text_upper[start: m_lbl.start() + 12000]
            for m_fmt in re.finditer(CNPJ_FORMATADO, bloco):
                c_raw = m_fmt.group(1)
                if _cnpj_is_valid(_cnpj_digits(c_raw)):
                    cands.append((start + m_fmt.start(), c_raw, f"CNPJ perto de '{lbl}' (±)"))
            for m_gen in re.finditer(CNPJ_GENERIC, bloco):
                c_raw = m_gen.group(1)
                if _cnpj_is_valid(_cnpj_digits(c_raw)):
                    cands.append((start + m_gen.start(), c_raw, f"CNPJ genérico perto de '{lbl}' (±)"))
    for m_all in re.finditer(CNPJ_GENERIC, text_upper):
        c_raw = m_all.group(1); d = _cnpj_digits(c_raw)
        if _cnpj_is_valid(d): cands.append((m_all.start(), c_raw, "CNPJ por varredura"))
    seen, uniq = set(), []
    for pos, c_raw, src in cands:
        d = _cnpj_digits(c_raw)
        if d not in seen:
            seen.add(d); uniq.append((pos, c_raw, src))
    return uniq

def _pick_best_dest_cnpj(text_upper: str, candidates: list) -> tuple:
    if not candidates: return None, None
    pos_hints  = ("DESTINAT", "REMETENTE", "LOCAL DE ENTREGA", "TOMADOR")
    house_hints = ("PAGUE MENOS", "EMPREEN")
    neg_hints  = ("TRANSPORTADOR", "FRETE POR CONTA", "EMITENTE", "PRESTADOR", "PROTOCOLO", "CHAVE DE ACESSO")
    best = None; best_score = -10**9; best_src = None
    for pos, c_raw, src in candidates:
        d   = _cnpj_digits(c_raw)
        win = text_upper[max(0, pos-2000): pos+2000]
        s = 0
        if d.startswith("04"): s += 3
        if d.startswith("06"): s += 2
        if any(lbl in win for lbl in pos_hints):  s += 6
        if any(lbl in win for lbl in house_hints): s += 4
        if any(lbl in win for lbl in neg_hints):  s -= 5
        if s > best_score:
            best_score = s; best = c_raw; best_src = f"{src} (score={s})"
    return best, best_src

# ======================================================================
# Extratores
# ======================================================================
def extrair_vencimento_de_texto(texto: str) -> str or None:
    if not texto: return None
    t = _normalize_text(texto).upper()
    m = re.search(r"\b(VENC(?:IMENTO)?|VENCTO|VCTO)\b[^0-9]{0,10}(\d{2}/\d{2}/\d{4})", t)
    if m: return m.group(2)
    m2 = re.search(r"\b(\d{2}/\d{2}/\d{4})\b", t)
    return m2.group(1) if m2 else None

def _find_oc_candidates(text_upper: str) -> list:
    oc_pattern = r"\b(45\d{8,10})\b"
    cands = []
    labels = [
        "ORDEM DE COMPRA", "OC", "PEDIDO", "PEDIDO CLIENTE", "PEDIDO DE VENDAS",
        "DADOS ADICIONAIS", "INFORMAÇÕES COMPLEMENTARES", "DISCRIMINAÇÃO DOS SERVIÇOS",
    ]
    for lbl in labels:
        m_lbl = re.search(lbl, text_upper)
        if m_lbl:
            start = max(0, m_lbl.start()-2000)
            bloco = text_upper[start: m_lbl.start()+4000]
            for m in re.finditer(oc_pattern, bloco):
                cands.append((start + m.start(), m.group(1), 5))
    for m in re.finditer(oc_pattern, text_upper):
        cands.append((m.start(), m.group(1), 1))
    seen, uniq = set(), []
    for pos, oc, score in cands:
        if oc not in seen:
            seen.add(oc); uniq.append((pos, oc, score))
    return uniq

def extrair_dados_pdf(caminho_pdf, debug=False):
    """
    Extrai dados de NF-e e NFS-e com regex melhorados para evitar capturar
    a chave de acesso como número de nota
    """
    if PdfReader is None:
        return None, None, None, None, None, None, {}
    
    try:
        reader = PdfReader(caminho_pdf)
        texto_completo = ""
        for page in reader.pages:
            texto_completo += (page.extract_text() or "") + "\n"
        
        t_upper = texto_completo.upper()
        is_nfse = "NFS-E" in t_upper or "SERVIÇO" in t_upper or "TOMADOR" in t_upper

        # ==================== CNPJ E LOJA ====================
        cands = _find_cnpj_candidates(t_upper)
        cnpj_dest_raw, src_dest = _pick_best_dest_cnpj(t_upper, cands)
        cnpj_dest = _cnpj_digits(cnpj_dest_raw or "")
        loja, origem_loja = _derive_loja_from_cnpj(cnpj_dest, src_dest or "não encontrado")

        # Fallback: procura loja em texto se não encontrou no CNPJ
        if not loja:
            loja_bloco = None
            
            # Se encontrou CNPJ, procura perto dele
            if cnpj_dest_raw:
                pos = t_upper.find(cnpj_dest_raw)
                if pos >= 0:
                    loja_bloco = t_upper[max(0, pos-1500): pos+2500]
            
            # Se não, procura no bloco DESTINATÁRIO/TOMADOR
            if not loja_bloco:
                m_dest = re.search(
                    r"((?:DESTINAT[ÁA]RIO|TOMADOR)[^\n]{0,3000})", 
                    t_upper
                )
                loja_bloco = m_dest.group(1) if m_dest else t_upper
            
            # Padrões de busca de loja (em ordem de prioridade)
            padroes_loja = [
                r"(?:LOJA|FILIAL|LJ|FL)[^\d]{0,15}(\d{1,5})\b",
                r"PAGUE\s*MENOS[^\d]{0,25}(\d{1,5})\b",
                r"(?:CD|LOJA)\s+[A-Z]{2}[/\s-]+([A-Z0-9]{2,5})",
            ]
            
            for padrao in padroes_loja:
                m_lj = re.search(padrao, loja_bloco)
                if m_lj:
                    # Remove letras se houver
                    loja_raw = m_lj.group(1)
                    # Extrai apenas números
                    nums = re.findall(r"\d+", loja_raw)
                    if nums:
                        loja = nums[0]
                        origem_loja = f"regex: {padrao[:30]}"
                        break

        # ==================== NÚMERO DA NF ====================
        nf, serie = None, None
        
        # Padrão 1: NFS-e com formato "NÚMERO/SÉRIE"
        if is_nfse:
            m_belem = re.search(r"N[ÚU]MERO\s*/\s*S[ÉE]RIE\s*(\d+)\s*/\s*([A-Z0-9]+)", t_upper)
            if m_belem: 
                nf, serie = m_belem.group(1), m_belem.group(2)
        
        # Padrão 2: "Número da NFS-e" seguido do número (formato Uberlândia)
        if not nf:
            m_nfse = re.search(r"N[ÚU]MERO\s+DA\s+NFS?-?E[\r\n\s:]*(\d{1,15})\b", t_upper)
            if m_nfse:
                candidato = m_nfse.group(1)
                # Rejeita se for chave de acesso (44 dígitos)
                if len(candidato) < 20:
                    nf = candidato
        
        # Padrão 3: "NÚMERO" genérico (evitando chave de acesso)
        if not nf:
            # Lista de contextos onde NÃO queremos pegar o número
            contextos_negativos = [
                "CHAVE", "PROTOCOLO", "AUTORIZACAO", "AUTORIZAÇÃO",
                "CONTROLE", "LOTE", "RECIBO"
            ]
            
            # Procura "NÚMERO" seguido de dígitos
            for m in re.finditer(
                r"(?:N[ÚU]MERO|NF(?!S)|NFS-?E)(?:\s+DA\s+NFS?-?E)?[^\d]{0,50}(\d{1,15})\b",
                t_upper
            ):
                candidato = m.group(1)
                
                # Rejeita se for muito longo (provavelmente chave)
                if len(candidato) > 15:
                    continue
                
                # Rejeita se for chave de acesso (44 dígitos)
                if len(candidato) == 44:
                    continue
                
                # Verifica contexto (100 caracteres antes)
                contexto = t_upper[max(0, m.start()-100):m.start()]
                
                # Rejeita se estiver em contexto negativo
                if any(neg in contexto for neg in contextos_negativos):
                    continue
                
                # Se passou todas as validações, aceita
                nf = candidato
                break
        
        # Padrão 4: Última tentativa - procura após "NOTA FISCAL"
        if not nf:
            m_nf_generico = re.search(
                r"(?:NOTA\s+FISCAL|DOCUMENTO\s+AUXILIAR)[^\d]{0,200}(\d{1,15})\b",
                t_upper
            )
            if m_nf_generico:
                candidato = m_nf_generico.group(1)
                if len(candidato) < 20:  # Não é chave
                    nf = candidato
        
        # ==================== SÉRIE ====================
        if not serie:
            m_s = re.search(r"S[ÉE]RIE(?:\s+DA\s+DPS)?[^\dA-Z]{0,50}(\d{1,5}|[A-Z]{1,3})\b", t_upper)
            if m_s: 
                serie = m_s.group(1)

        # ==================== OUTROS CAMPOS ====================
        valor_total_str = None
        if is_nfse:
            all_vals = re.findall(r"([\d\.]{1,12},\d{2})", t_upper)
            numeric_vals = []
            for v in all_vals:
                try: 
                    numeric_vals.append(float(v.replace(".", "").replace(",", ".")))
                except: 
                    pass
            if numeric_vals:
                max_val = max(numeric_vals)
                valor_total_str = f"{max_val:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
        
        if not valor_total_str:
            m_v = re.search(
                r"(VALOR\s*(?:TOTAL|L[ÍI]QUIDO|DA\s*NOTA|DOS\s*SERVI[ÇC]OS)|TOTAL\s*(?:A\s*PAGAR|GERAL))[^\d]{0,40}([\d\.,]{2,})",
                t_upper
            )
            if m_v: 
                valor_total_str = m_v.group(2)
        
        if not valor_total_str:
            m_v2 = re.search(r"TOTAL[^\d]{0,25}([\d\.,]{4,})", t_upper)
            if m_v2: 
                valor_total_str = m_v2.group(1)

        chave = (re.search(r"(?:CHAVE\s*DE\s*ACESSO|CHAVE)[^\d]{0,60}(\d{44})", t_upper) or [None, None])[1]
        cfop  = (re.search(r"CFOP[^\d]{0,60}(\d{4})", t_upper) or [None, None])[1]

        # Emitente
        emit_cnpj = None
        emit_nome = None
        for pos, c_raw, src in cands:
            d = _cnpj_digits(c_raw)
            if d and d != cnpj_dest:
                win = t_upper[max(0, pos-800): pos+800]
                if any(x in win for x in ["EMITENTE", "PRESTADOR", "NOME / NOME EMPRESARIAL", "IDENTIFICAÇÃO DO EMITENTE"]):
                    emit_cnpj = d
                    m_nome = re.search(r"(?:NOME\s*/\s*NOME\s*EMPRESARIAL|IDENTIFICAÇÃO DO EMITENTE)\s*([^\d\n]{3,120})", win)
                    if m_nome: 
                        emit_nome = m_nome.group(1).strip()
                    break
        
        if not emit_cnpj and cands:
            for pos, c_raw, src in cands:
                d = _cnpj_digits(c_raw)
                if d and d != cnpj_dest:
                    emit_cnpj = d
                    break

        # Ordem de Compra
        oc_cands = _find_oc_candidates(t_upper)
        oc = max(oc_cands, key=lambda x: x[2])[1] if oc_cands else None
        
        # Data de emissão
        data_emissao = (re.search(
            r"(?:DATA\s*(?:DE\s*|E\s*HORA\s*(?:DA\s*|DE\s*))?EMISS[ÃA]O)[^\d]{0,60}(\d{2}/\d{2}/\d{4})", 
            t_upper
        ) or [None, None])[1]

        # ==================== DEBUG ====================
        if debug:
            print(f"\n{'='*60}")
            print(f"ARQUIVO: {os.path.basename(caminho_pdf)}")
            print(f"{'='*60}")
            print(f"NF: {nf}")
            print(f"Série: {serie}")
            print(f"Loja: {loja}")
            print(f"Origem loja: {origem_loja}")
            print(f"CNPJ dest: {cnpj_dest}")
            print(f"Chave: {chave[:20] + '...' if chave else 'N/A'}")
            print(f"Valor: {valor_total_str}")
            print(f"{'='*60}\n")

        extras = {
            "Valor Total": valor_total_str,
            "ordem_compra": oc,
            "pedido": oc,
            "cfop": cfop,
            "emitente_cnpj": emit_cnpj,
            "emitente_nome": emit_nome,
            "destinatario_cnpj": cnpj_dest,
            "data_emissao": data_emissao,
            "data_vencimento": None,
        }
        
        return nf, loja, "extração PDF", origem_loja, serie, chave, extras

    except Exception as e:
        print(f"⚠️ Erro ao ler {os.path.basename(caminho_pdf)}: {e}")
        return None, None, None, None, None, None, {}

# ======================================================================
# Outlook multi-Inbox (Win32)
# ======================================================================
PR_SMTP_ADDRESS = "http://schemas.microsoft.com/mapi/proptag/0x39FE001E"

SUBJECT_PATTERNS = [
    r"\bnota\s*fiscal\b",
    r"\bnf-?e\b",
    r"\bdanfe\b",
    r"\bnf\b",
    r"\bnfs-?e\b",
    r"\bservi[cç]o\b",
]

def subject_is_nf(subject: str) -> bool:
    s = subject or ""
    for p in SUBJECT_PATTERNS:
        if re.search(p, s, re.IGNORECASE): return True
    return False

def _canon_email(e: str) -> str:
    e = (e or "").strip().lower()
    return e[5:] if e.startswith("smtp:") else e

def resolve_sender_smtp(mail_item):
    smtp = ""
    try:
        sender = getattr(mail_item, "Sender", None)
        if sender:
            ae = sender.AddressEntry
            if ae:
                pa = ae.PropertyAccessor
                try:
                    smtp = (pa.GetProperty(PR_SMTP_ADDRESS) or "").lower()
                except:
                    smtp = ""
        if not smtp:
            smtp = (getattr(mail_item, "SenderEmailAddress", "") or "").lower()
            if smtp.startswith("smtp:"): smtp = smtp[5:]
    except:
        smtp = (getattr(mail_item, "SenderEmailAddress", "") or "").lower()
    return smtp

def _match_sender_or_subject(sender_addr: str, allow_list: list, sender_name: str, subject: str) -> (bool, str):
    if not allow_list:
        if subject_is_nf(subject): return True, "sem filtro, match por assunto NF"
        return False, "sem filtro, assunto não indica NF"
    s = _canon_email(sender_addr)
    if s in allow_list: return True, "match exato smtp"
    if any(s.endswith(x) for x in allow_list): return True, "match por sufixo smtp"
    return False, f"sender '{s}' não está na lista"

def resolve_subfolder(root_folder, path_str):
    if not path_str: return root_folder
    cur = root_folder
    for part in path_str.split("/"):
        if not part: continue
        cur = cur.Folders(part)
    return cur

def get_all_candidate_inboxes(namespace, subpath=""):
    candidates = []
    try:
        stores = namespace.Stores
        for i in range(1, stores.Count + 1):
            st = stores.Item(i)
            try:
                inbox = st.GetDefaultFolder(6)  # Inbox
                target = resolve_subfolder(inbox, subpath)
                candidates.append(("STORE", st.DisplayName, target))
            except: pass
    except: pass
    try:
        for i in range(1, namespace.Session.Accounts.Count + 1):
            acc = namespace.Session.Accounts.Item(i)
            try:
                ds = acc.DeliveryStore
                inbox = ds.GetDefaultFolder(6)
                target = resolve_subfolder(inbox, subpath)
                candidates.append(("ACCOUNT", acc.SmtpAddress, target))
            except: pass
    except: pass
    try:
        inbox = namespace.GetDefaultFolder(6)
        target = resolve_subfolder(inbox, subpath)
        candidates.append(("DEFAULT", "DefaultStore", target))
    except: pass

    seen, uniq = set(), []
    for kind, name, folder in candidates:
        try: fp = folder.FolderPath
        except: fp = f"{kind}:{name}"
        if fp not in seen:
            seen.add(fp); uniq.append((kind, name, folder))
    return uniq

def coletar_anexos_outlook_multi(filter_senders_lower):
    if win32 is None:
        print("❌ pywin32 não disponível ou Outlook não é o Clássico (Win32).")
        return []
    outlook = win32.Dispatch("Outlook.Application").GetNamespace("MAPI")

    dt_from_naive = (dt.datetime.now().replace(tzinfo=None) - dt.timedelta(days=LOOKBACK_DAYS))
    inboxes = get_all_candidate_inboxes(outlook, SUBPASTA_PATH.replace("\\", "/"))

    print("\n🧭 Inboxes candidatas localizadas:")
    for kind, name, folder in inboxes:
        try: print(f"   - {kind}: {name} | {folder.FolderPath}")
        except: print(f"   - {kind}: {name}")

    coletados, ignorados = [], []

    for kind, name, folder in inboxes:
        try:
            items = folder.Items
            items.Sort("[ReceivedTime]", True)  # DESC
            it = items.GetFirst()
            i_debug = 0
            while it:
                try:
                    received = getattr(it, "ReceivedTime", None)
                    received_naive = _to_naive_local(received)
                    if received_naive and received_naive < dt_from_naive: break
                    if not getattr(it, "Attachments", None):
                        it = items.GetNext(); continue

                    sender_name = (getattr(it, "SenderName", "") or "").strip()
                    sender_addr = resolve_sender_smtp(it)
                    subject = (getattr(it, "Subject", "") or "")
                    body = getattr(it, "Body", "")

                    match, motivo = _match_sender_or_subject(sender_addr, filter_senders_lower, sender_name, subject)
                    found_pdf = any(str(att.FileName or "").lower().endswith(".pdf") for att in it.Attachments)

                    if not match or not found_pdf:
                        ignorados.append({
                            "Inbox": getattr(folder, "FolderPath", f"{kind}:{name}"),
                            "Sender": sender_addr,
                            "SenderName": sender_name,
                            "Subject": subject[:200],
                            "Received": received_naive,
                            "TemPDF": found_pdf,
                            "Motivo": motivo if not match else "sem PDF"
                        })
                        it = items.GetNext(); continue

                    # Salvar PDFs em _temp local; depois movemos para a pasta do projeto/fornecedor
                    temp_dir = os.path.join(os.getcwd(), "_temp_nfs"); _ensure_dirs(temp_dir)
                    for att in it.Attachments:
                        fname = str(att.FileName or "")
                        if not fname.lower().endswith(".pdf"): continue
                        temp_path = os.path.join(temp_dir, _sanitize_filename(fname))
                        att.SaveAsFile(temp_path)
                        coletados.append({
                            "entry_id": it.EntryID,
                            "attach_id": f"{it.EntryID}::{fname}",
                            "sender_email": sender_addr,
                            "sender_name": sender_name,
                            "subject": subject,
                            "received": received_naive,
                            "body": body,
                            "temp_path": temp_path,
                            "orig_filename": fname,
                            "inbox_path": getattr(folder, "FolderPath", f"{kind}:{name}")
                        })

                    if DIAGNOSTICO and i_debug < 5:
                        print(f"   • {folder.FolderPath} | {received_naive} | From={sender_addr} | PDF=SIM | Subj={subject[:80]}")
                        i_debug += 1

                except Exception as e:
                    print(f"⚠️ Erro ao ler item ({getattr(folder,'FolderPath','')}): {e}")
                it = items.GetNext()
        except Exception as e:
            print(f"⚠️ Erro ao acessar pasta {getattr(folder,'FolderPath','')}: {e}")

    if DIAGNOSTICO and LOG_IGNORADOS and ignorados:
        print("\n📝 E-mails ignorados:")
        for ig in ignorados[:30]:
            print(f"   - {ig['Inbox']} | {ig['Received']} | From={ig['Sender']} | PDF={ig['TemPDF']} | Subj={ig['Subject']} | Motivo={ig['Motivo']}")
    return coletados

# ======================================================================
# MAIN
# ======================================================================
def main():
    # 1) Selecionar PROJETO
    projeto = select_projeto_menu(PROJETOS)
    if not projeto:
        return

    # 2) Digitar FORNECEDOR
    fornecedor = ask_fornecedor_nome()

    # 3) Montar caminhos
    pasta_projeto = os.path.join(BASE_PROJETOS, projeto)
    pasta_fornecedor = os.path.join(pasta_projeto, fornecedor)

    if not os.path.isdir(pasta_projeto):
        print(f"\n❌ A pasta do PROJETO não existe:\n   {pasta_projeto}\nCrie-a e rode novamente.")
        return

    if not os.path.isdir(pasta_fornecedor):
        print(f"\n❌ A pasta do FORNECEDOR não existe dentro do projeto:\n   {pasta_fornecedor}\n"
              f"Como solicitado, o script **NÃO** cria pastas automaticamente.")
        seguir = input("Deseja seguir apenas para gerar o Excel (sem mover PDFs)? [s/N]: ").strip().lower()
        if seguir not in ("s", "sim", "y", "yes"):
            return

    # 4) Informar remetentes (SMTP) a filtrar (ou vazio para TODOS com assunto NF/NFS-e/DANFE)
    raw_emails = input("\nDigite os e-mails de remetentes a filtrar (separados por vírgula; deixe vazio para TODOS): ").strip()
    FILTER_SENDERS = [e.strip().lower() for e in raw_emails.split(",") if e.strip()]
    if FILTER_SENDERS:
        print("✅ Filtrando por remetentes:", FILTER_SENDERS)
    else:
        print("ℹ️ Sem filtro de remetente: match por assunto (NF/NFS-e/DANFE).")

    # 5) Coletar anexos
    registros = coletar_anexos_outlook_multi(FILTER_SENDERS)
    print(f"\n📥 PDFs encontrados: {len(registros)}")

    # 6) Processar PDFs (mover somente se pasta do fornecedor existir)
    base_registros = []
    movidos = 0

    for reg in registros:
        pdf_path = reg["temp_path"]
        nf, loja, origem_nf, origem_loja, serie, chave, extras = extrair_dados_pdf(pdf_path, debug=False)

        if nf and loja:
            novo_nome = f"NF {nf} - LJ {loja}.pdf"
        elif nf:
            novo_nome = f"NF {nf}.pdf"
        else:
            base_nome = os.path.splitext(reg["orig_filename"])[0]
            novo_nome = _sanitize_filename(f"{base_nome} - PROCESSADO.pdf")

        destino_final = os.path.join(pasta_fornecedor, novo_nome)

        status = "SEM PASTA FORNECEDOR"
        if os.path.isdir(pasta_fornecedor):
            try:
                if os.path.exists(destino_final):
                    h = _md5_file(pdf_path)[:8]
                    destino_final = os.path.join(pasta_fornecedor, f"{os.path.splitext(novo_nome)[0]} - {h}.pdf")
                shutil.move(pdf_path, destino_final)
                status = "OK"
                movidos += 1
                print(f"✅ {os.path.basename(destino_final)}  ←  {reg.get('inbox_path','')}")
            except Exception as e:
                print(f"❌ Falha ao mover {pdf_path} -> {destino_final}: {e}")
                status = f"ERRO: {e}"
        else:
            print(f"⚠️ Pasta do fornecedor ausente, não movido: {novo_nome}")

        data_venc = extras.get("data_vencimento") or \
                    (subject_is_nf(reg["subject"]) and extrair_vencimento_de_texto(reg["subject"])) or \
                    extrair_vencimento_de_texto(reg["body"])

        base_registros.append({
            "Projeto": projeto,
            "Fornecedor": fornecedor,
            "Arquivo": os.path.basename(destino_final) if os.path.isdir(pasta_fornecedor) else novo_nome,
            "Caminho": destino_final if os.path.isdir(pasta_fornecedor) else "(não movido - sem pasta fornecedor)",
            "NF": nf,
            "Loja": loja,
            "Série": serie,
            "Chave": chave,
            "Valor Total": extras.get("Valor Total"),
            "Pedido": extras.get("pedido"),
            "Ordem de Compra": extras.get("ordem_compra"),
            "CFOP": extras.get("cfop"),
            "Emitente CNPJ": extras.get("emitente_cnpj"),
            "Destinatário CNPJ": extras.get("destinatario_cnpj"),
            "Data Emissão": extras.get("data_emissao"),
            "Vencimento": data_venc,
            "Assunto Email": reg["subject"],
            "Remetente": reg["sender_email"],
            "Remetente Nome": reg["sender_name"],
            "Recebido Em": reg["received"],
            "Inbox": reg.get("inbox_path", ""),
            "Status": status
        })

    # 7) Snapshot (execução atual) -> salva na pasta do PROJETO
    if base_registros and pd is not None:
        df_run = pd.DataFrame(base_registros)
        if "Recebido Em" in df_run.columns:
            df_run = df_run.sort_values(by="Recebido Em", ascending=False, ignore_index=True)

        # ======== AJUSTE CRÍTICO: agregação com rótulo 'Valor_Total_R$' ========
        work = df_run.copy()
        work["ValorNum"] = work["Valor Total"].apply(_parse_brl_to_float)

        # Usamos named aggregation com **{...} para permitir 'R$' no nome
        resumo = work.groupby(["Fornecedor","Status"], dropna=False).agg(
            Qtde_NFs=("NF", "count"),
            Qtde_com_Loja=("Loja", lambda s: s.notna().sum()),
            **{"Valor_Total_R$": ("ValorNum", "sum")},
            Primeiro_Recebido=("Recebido Em", "min"),
            Ultimo_Recebido=("Recebido Em", "max"),
        ).reset_index()

        resumo["Qtde_sem_Loja"] = resumo["Qtde_NFs"] - resumo["Qtde_com_Loja"]
        resumo = resumo.sort_values(["Status","Fornecedor"], ignore_index=True)

        def fmt_brl(x):
            try: return f"R$ {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
            except: return "R$ 0,00"

        print("\n📈 RESUMO (execução atual):\n")
        rprint = resumo.copy()
        rprint["Valor_Total_R$"] = rprint["Valor_Total_R$"].map(fmt_brl)
        print(rprint.to_string(index=False))

        hoje = dt.datetime.now().strftime("%Y-%m-%d")
        snap_path = os.path.join(pasta_projeto, f"BASE_NFs_{fornecedor}_{hoje}.xlsx")
        with pd.ExcelWriter(snap_path, engine="openpyxl") as writer:
            df_run.to_excel(writer, index=False, sheet_name="BASE")
            resumo.to_excel(writer, index=False, sheet_name="RESUMO")
        print(f"\n🗂️ Snapshot salvo em: {snap_path}")

    # 8) Limpa _temp se vazio
    temp_dir = os.path.join(os.getcwd(), "_temp_nfs")
    if os.path.isdir(temp_dir) and not os.listdir(temp_dir):
        try: os.rmdir(temp_dir)
        except: pass

    print(f"\n✨ Concluído. PDFs movidos: {movidos}")

# ---- RUN ----
if __name__ == "__main__":
    if not os.path.isdir(BASE_PROJETOS):
        print(f"❌ Caminho BASE dos projetos não existe:\n   {BASE_PROJETOS}")
    elif win32 is None:
        print("\n❌ pywin32 não disponível ou Outlook não é o Clássico (Win32).")
        print("   Garanta: Outlook Clássico aberto/logado + 'pip install pywin32 pandas openpyxl PyPDF2'\n")
    else:
        main()


📁 Selecione o PROJETO:
 1. ESPECIAL
 2. LAUDO DE RISCO
 3. LAUDO DE RISCO CD
 4. LAUDO DE RISCO MATRIZ
 5. LOJAS NOVAS
 6. MISSOES HIGIENICAS
 7. OUTROS
 8. PROJETO ESPECIAL
 9. RESIZE
 10. RETROFIT
 11. VIRADA DE BANDEIRA
 0. Cancelar


Digite o número da opção:  11


✅ Selecionado: VIRADA DE BANDEIRA



Digite o NOME exato do FORNECEDOR (ex.: LUMICENTER):  PLANAC

Digite os e-mails de remetentes a filtrar (separados por vírgula; deixe vazio para TODOS):  arnaldo.silva@sofkym.com


✅ Filtrando por remetentes: ['arnaldo.silva@sofkym.com']

🧭 Inboxes candidatas localizadas:
   - STORE: tiagooliveira@pmenos.com.br | \\tiagooliveira@pmenos.com.br\Caixa de Entrada
   • \\tiagooliveira@pmenos.com.br\Caixa de Entrada | 2026-01-28 16:14:08.990000 | From=arnaldo.silva@sofkym.com | PDF=SIM | Subj=Re: Pedidos Criados - Fornecedor DWA - Coord HENRIQUE

📥 PDFs encontrados: 14
✅ NF 98 - LJ 7045.pdf  ←  \\tiagooliveira@pmenos.com.br\Caixa de Entrada
✅ NF 98 - LJ 7126.pdf  ←  \\tiagooliveira@pmenos.com.br\Caixa de Entrada
✅ NF 98 - LJ 7129.pdf  ←  \\tiagooliveira@pmenos.com.br\Caixa de Entrada
✅ NF 98 - LJ 7054.pdf  ←  \\tiagooliveira@pmenos.com.br\Caixa de Entrada
✅ NF 98 - LJ 7050.pdf  ←  \\tiagooliveira@pmenos.com.br\Caixa de Entrada
✅ NF 98 - LJ 7132.pdf  ←  \\tiagooliveira@pmenos.com.br\Caixa de Entrada
✅ NF 98 - LJ 7289.pdf  ←  \\tiagooliveira@pmenos.com.br\Caixa de Entrada
✅ NF 98 - LJ 7174.pdf  ←  \\tiagooliveira@pmenos.com.br\Caixa de Entrada
✅ NF 98 - LJ 7127.pdf  ←  \

In [1]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

import os, re, sys, csv, hashlib, shutil, datetime as dt

# --------- Dependências críticas ---------
missing = []
try:
    import win32com.client as win32
except Exception:
    win32 = None
    missing.append("pywin32")
try:
    import pandas as pd
except Exception:
    pd = None
    missing.append("pandas")
try:
    from PyPDF2 import PdfReader
except Exception:
    PdfReader = None
    missing.append("PyPDF2")
try:
    import openpyxl
except Exception:
    missing.append("openpyxl")

if missing:
    print("⚠️ Dependências ausentes:", ", ".join(missing))
    print("   Instale no Anaconda Prompt:")
    if "pywin32" in missing: print("   pip install pywin32")
    if "pandas" in missing: print("   pip install pandas")
    if "openpyxl" in missing: print("   pip install openpyxl")
    if "PyPDF2" in missing: print("   pip install PyPDF2")

# ======================================================================
# Parâmetros fixos do seu ambiente
# ======================================================================
BASE_PROJETOS = r"C:\Users\126815\OneDrive - paguemenos.com.br\Área de Trabalho\NFS\2025"

PROJETOS = [
    "ESPECIAL", "LAUDO DE RISCO", "LAUDO DE RISCO CD", "LAUDO DE RISCO MATRIZ",
    "LOJAS NOVAS", "MISSOES HIGIENICAS", "OUTROS", "PROJETO ESPECIAL",
    "RESIZE", "RETROFIT", "VIRADA DE BANDEIRA",
]

LOOKBACK_DAYS = 15
SUBPASTA_PATH = ""       # ex.: "Engenharia/NFs"
DIAGNOSTICO = True
LOG_IGNORADOS = False

# ======================================================================
# UI Terminal
# ======================================================================
def select_projeto_menu(opcoes: list) -> str:
    print("\n📁 Selecione o PROJETO:")
    for i, nome in enumerate(opcoes, start=1):
        print(f" {i}. {nome}")
    print(" 0. Cancelar")
    while True:
        try:
            op = int(input("Digite o número da opção: ").strip())
            if op == 0: return None
            if 1 <= op <= len(opcoes):
                projeto = opcoes[op - 1]
                print(f"✅ Selecionado: {projeto}")
                return projeto
        except ValueError: pass
        print("Opção inválida. Tente novamente.")

def ask_fornecedor_nome() -> str:
    nome = input("\nDigite o NOME exato do FORNECEDOR (ex.: LUMICENTER): ").strip()
    while not nome:
        nome = input("Digite o NOME do FORNECEDOR: ").strip()
    return nome

# ======================================================================
# Utils locais
# ======================================================================
def _ensure_dirs(path): os.makedirs(path, exist_ok=True)

def _sanitize_filename(name: str) -> str:
    name = (name or "").strip().replace("\n", " ").replace("\r", " ")
    name = re.sub(r'[<>:"/\\|?*]', "_", name)
    name = re.sub(r"\s{2,}", " ", name)
    return name

def _md5_file(path: str) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""): h.update(chunk)
    return h.hexdigest()

def _normalize_text(text: str) -> str:
    if not text: return ""
    t = re.sub(r"[\n\r\t]+", " ", text)
    t = re.sub(r"\s{2,}", " ", t)
    return t.upper()

def _parse_brl_to_float(v):
    if v is None: return None
    s = str(v)
    s = re.sub(r"[Rr]\$\s*", "", s)
    s = s.replace(".", "TEMP").replace(",", ".").replace("TEMP", "")
    s = re.sub(r"[^\d\.]", "", s)
    try: return float(s)
    except: return None

def _to_naive_local(d: dt.datetime) -> dt.datetime or None:
    if d is None: return None
    try:
        if getattr(d, "tzinfo", None) is not None:
            return d.astimezone().replace(tzinfo=None)
        return d
    except Exception:
        try: return dt.datetime.fromtimestamp(d.timestamp()).replace(tzinfo=None)
        except Exception: return None

# ======================================================================
# CNPJ & Loja
# ======================================================================
def _cnpj_digits(s: str) -> str:
    d = re.sub(r"\D", "", s or "")
    return d if len(d) == 14 else ""

def _cnpj_is_valid(d: str) -> bool:
    d = _cnpj_digits(d)
    if len(d) != 14 or len(set(d)) == 1: return False
    def dv_calc(digs, pesos):
        soma = sum(int(digs[i]) * pesos[i] for i in range(len(pesos)))
        r = soma % 11
        return '0' if r < 2 else str(11 - r)
    base = d[:12]
    dv1 = dv_calc(base, [5,4,3,2,9,8,7,6,5,4,3,2])
    dv2 = dv_calc(base + dv1, [6,5,4,3,2,9,8,7,6,5,4,3,2])
    return d[-2:] == dv1 + dv2

def _derive_loja_from_cnpj(cnpj_raw: str, origem_cnpj: str) -> tuple:
    d = _cnpj_digits(cnpj_raw)
    if not d: return None, origem_cnpj
    seq = d[8:12]
    origem_loja = origem_cnpj
    if d.startswith("04"):
        loja = "7" + seq[1:]; origem_loja += " + Regra IMIFARMA"
    elif d.startswith("06"):
        loja = seq.lstrip("0") or "0"; origem_loja += " + Regra Pague Menos"
    else:
        loja = seq.lstrip("0") or seq
    return loja, origem_loja

# ======================================================================
# Extratores
# ======================================================================
def extrair_vencimento_de_texto(texto: str) -> str or None:
    if not texto: return None
    t = _normalize_text(texto).upper()
    m = re.search(r"\b(VENC(?:IMENTO)?|VENCTO|VCTO)\b[^0-9]{0,10}(\d{2}/\d{2}/\d{4})", t)
    if m: return m.group(2)
    m2 = re.search(r"\b(\d{2}/\d{2}/\d{4})\b", t)
    return m2.group(1) if m2 else None

def _find_oc_candidates(text_upper: str) -> list:
    oc_pattern = r"\b(45\d{8,10})\b"
    cands = []
    labels = ["ORDEM DE COMPRA", "OC", "PEDIDO", "PEDIDO CLIENTE", "DADOS ADICIONAIS", "INFORMAÇÕES COMPLEMENTARES"]
    for lbl in labels:
        m_lbl = re.search(lbl, text_upper)
        if m_lbl:
            start = max(0, m_lbl.start()-2000)
            bloco = text_upper[start: m_lbl.start()+4000]
            for m in re.finditer(oc_pattern, bloco):
                cands.append((start + m.start(), m.group(1), 5))
    for m in re.finditer(oc_pattern, text_upper):
        cands.append((m.start(), m.group(1), 1))
    seen, uniq = set(), []
    for pos, oc, score in cands:
        if oc not in seen:
            seen.add(oc); uniq.append((pos, oc, score))
    return uniq

def extrair_dados_pdf(caminho_pdf, debug=False):
    if PdfReader is None: return None, None, None, None, None, None, {}
    try:
        reader = PdfReader(caminho_pdf)
        texto_completo = "\n".join([p.extract_text() or "" for p in reader.pages])
        t_upper = texto_completo.upper()
        
        # --- CNPJ E LOJA ---
        cnpj_dest = None
        m_cnpj = re.search(r"(\d{2}\.\d{3}\.\d{3}/\d{4}\-\d{2})", t_upper)
        if m_cnpj: cnpj_dest = _cnpj_digits(m_cnpj.group(1))
        loja, origem_loja = _derive_loja_from_cnpj(cnpj_dest, "CNPJ")
        
        if not loja:
            m_lj = re.search(r"(?:LOJA|LJ|FL)[^\d]{0,10}(\d{1,5})\b", t_upper)
            if m_lj: loja = m_lj.group(1)

        # --- NÚMERO DA NF (LÓGICA CORRIGIDA PARA EVITAR 98) ---
        nf = None
        # 1. Tentar capturar o número que vem no final da Inscrição Municipal (comum em São Luís)
        m_insc = re.search(r"INSCRIÇÃO\s+MUNICIPAL:\s*\d{8,}(\d{4,7})\b", t_upper)
        if m_insc:
            nf = m_insc.group(1)
        
        # 2. Tentar padrões de rótulo, mas filtrando o (98) e TELEFONE
        if not nf:
            padroes = [
                r"N[ÚU]MERO\s+DA\s+NOTA[\r\n\s:]*(\d+)",
                r"N[ÚU]MERO\s+DA\s+NFS?-?E[\r\n\s:]*(\d+)",
                r"NF-E\s+N[º°\.\s]*(\d+)"
            ]
            for p in padroes:
                for m in re.finditer(p, t_upper):
                    cand = m.group(1)
                    contexto = t_upper[max(0, m.start()-5):m.end()+20]
                    if "(98)" in contexto or "TELEFONE" in contexto: continue
                    if 1 < len(cand) < 10:
                        nf = cand
                        break
                if nf: break

        # --- OUTROS CAMPOS ---
        serie = (re.search(r"S[ÉE]RIE[^\dA-Z]{0,10}(\d+|[A-Z]+)\b", t_upper) or [None, None])[1]
        valor = (re.search(r"VALOR\s*TOTAL[^\d]{0,20}([\d\.,]{4,})", t_upper) or [None, None])[1]
        chave = (re.search(r"CHAVE[^\d]{0,20}(\d{44})", t_upper) or [None, None])[1]
        oc_cands = _find_oc_candidates(t_upper)
        oc = max(oc_cands, key=lambda x: x[2])[1] if oc_cands else None
        data_emissao = (re.search(r"(?:DATA\s*(?:DE\s*|E\s*HORA\s*(?:DA\s*|DE\s*))?EMISS[ÃA]O)[^\d]{0,60}(\d{2}/\d{2}/\d{4})", t_upper) or [None, None])[1]

        extras = {
            "Valor Total": valor,
            "ordem_compra": oc,
            "pedido": oc,
            "data_emissao": data_emissao,
            "destinatario_cnpj": cnpj_dest
        }
        return nf, loja, "PDF", origem_loja, serie, chave, extras
    except Exception as e:
        if debug: print(f"Erro: {e}")
        return None, None, None, None, None, None, {}

# ======================================================================
# Outlook (Win32)
# ======================================================================
PR_SMTP_ADDRESS = "http://schemas.microsoft.com/mapi/proptag/0x39FE001E"
SUBJECT_PATTERNS = [r"\bnota\s*fiscal\b", r"\bnf-?e\b", r"\bdanfe\b", r"\bnf\b", r"\bnfs-?e\b"]

def subject_is_nf(subject: str) -> bool:
    s = subject or ""
    return any(re.search(p, s, re.IGNORECASE) for p in SUBJECT_PATTERNS)

def resolve_sender_smtp(mail_item):
    try:
        sender = mail_item.Sender
        if sender:
            return (sender.AddressEntry.PropertyAccessor.GetProperty(PR_SMTP_ADDRESS) or "").lower()
    except: pass
    return (getattr(mail_item, "SenderEmailAddress", "") or "").lower()

def resolve_subfolder(root_folder, path_str):
    if not path_str: return root_folder
    cur = root_folder
    for part in path_str.split("/"):
        if part: cur = cur.Folders(part)
    return cur

def coletar_anexos_outlook_multi(filter_senders_lower):
    if win32 is None: return []
    outlook = win32.Dispatch("Outlook.Application").GetNamespace("MAPI")
    dt_from = (dt.datetime.now() - dt.timedelta(days=LOOKBACK_DAYS))
    
    coletados = []
    try:
        inbox = outlook.GetDefaultFolder(6)
        folder = resolve_subfolder(inbox, SUBPASTA_PATH.replace("\\", "/"))
        items = folder.Items
        items.Sort("[ReceivedTime]", True)
        it = items.GetFirst()
        while it:
            try:
                received = _to_naive_local(getattr(it, "ReceivedTime", None))
                if received and received < dt_from: break
                
                sender_addr = resolve_sender_smtp(it)
                subject = getattr(it, "Subject", "")
                
                # Filtro de remetente ou assunto
                match = not filter_senders_lower or any(s in sender_addr for s in filter_senders_lower)
                if match or subject_is_nf(subject):
                    temp_dir = os.path.join(os.getcwd(), "_temp_nfs"); _ensure_dirs(temp_dir)
                    for att in it.Attachments:
                        if str(att.FileName).lower().endswith(".pdf"):
                            temp_path = os.path.join(temp_dir, _sanitize_filename(att.FileName))
                            att.SaveAsFile(temp_path)
                            coletados.append({
                                "sender_email": sender_addr,
                                "subject": subject,
                                "received": received,
                                "temp_path": temp_path,
                                "orig_filename": att.FileName
                            })
            except: pass
            it = items.GetNext()
    except Exception as e: print(f"Erro Outlook: {e}")
    return coletados

# ======================================================================
# MAIN
# ======================================================================
def main():
    projeto = select_projeto_menu(PROJETOS)
    if not projeto: return
    fornecedor = ask_fornecedor_nome()
    
    pasta_projeto = os.path.join(BASE_PROJETOS, projeto)
    pasta_fornecedor = os.path.join(pasta_projeto, fornecedor)
    
    if not os.path.isdir(pasta_projeto):
        print(f"❌ Pasta do projeto não existe: {pasta_projeto}"); return

    raw_emails = input("\nFiltro de e-mails (vazio para todos): ").strip()
    FILTER_SENDERS = [e.strip().lower() for e in raw_emails.split(",") if e.strip()]
    
    registros = coletar_anexos_outlook_multi(FILTER_SENDERS)
    print(f"📥 PDFs encontrados: {len(registros)}")
    
    base_registros = []
    for reg in registros:
        nf, loja, _, origem_lo, serie, chave, extras = extrair_dados_pdf(reg["temp_path"])
        
        novo_nome = f"NF {nf} - LJ {loja}.pdf" if nf and loja else f"NF {nf or 'SEM_NUM'}.pdf"
        destino = os.path.join(pasta_fornecedor, novo_nome)
        
        status = "OK"
        if os.path.isdir(pasta_fornecedor):
            try:
                if os.path.exists(destino):
                    destino = os.path.join(pasta_fornecedor, f"{os.path.splitext(novo_nome)[0]}_{_md5_file(reg['temp_path'])[:5]}.pdf")
                shutil.move(reg["temp_path"], destino)
            except Exception as e: status = f"Erro: {e}"
        else: status = "Pasta Fornecedor Ausente"

        base_registros.append({
            "Projeto": projeto, "Fornecedor": fornecedor, "NF": nf, "Loja": loja,
            "Valor Total": extras.get("Valor Total"), "Pedido": extras.get("pedido"),
            "Recebido Em": reg["received"], "Status": status, "Arquivo": os.path.basename(destino)
        })

    if base_registros and pd:
        df = pd.DataFrame(base_registros)
        hoje = dt.datetime.now().strftime("%Y-%m-%d")
        excel_path = os.path.join(pasta_projeto, f"BASE_NFs_{fornecedor}_{hoje}.xlsx")
        df.to_excel(excel_path, index=False)
        print(f"✅ Excel gerado: {excel_path}")

if __name__ == "__main__":
    main()


⚠️ Dependências ausentes: PyPDF2
   Instale no Anaconda Prompt:
   pip install PyPDF2

📁 Selecione o PROJETO:
 1. ESPECIAL
 2. LAUDO DE RISCO
 3. LAUDO DE RISCO CD
 4. LAUDO DE RISCO MATRIZ
 5. LOJAS NOVAS
 6. MISSOES HIGIENICAS
 7. OUTROS
 8. PROJETO ESPECIAL
 9. RESIZE
 10. RETROFIT
 11. VIRADA DE BANDEIRA
 0. Cancelar


Digite o número da opção:  5


✅ Selecionado: LOJAS NOVAS



Digite o NOME exato do FORNECEDOR (ex.: LUMICENTER):  HTB

Filtro de e-mails (vazio para todos):  Beatriz.Cikada@htb.eng.br


📥 PDFs encontrados: 11
✅ Excel gerado: C:\Users\126815\OneDrive - paguemenos.com.br\Área de Trabalho\NFS\2025\LOJAS NOVAS\BASE_NFs_HTB_2026-02-24.xlsx


In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

import os, re, sys, csv, hashlib, shutil, datetime as dt
import unicodedata

# --------- Dependências críticas ---------
missing = []
try:
    import win32com.client as win32
except Exception:
    win32 = None
    missing.append("pywin32")
try:
    import pandas as pd
except Exception:
    pd = None
    missing.append("pandas")
try:
    from PyPDF2 import PdfReader
except Exception:
    PdfReader = None
    missing.append("PyPDF2")
try:
    import openpyxl
except Exception:
    missing.append("openpyxl")

if missing:
    print("⚠️ Dependências ausentes:", ", ".join(missing))
    print("   Instale no Anaconda Prompt:")
    if "pywin32" in missing: print("   pip install pywin32")
    if "pandas" in missing: print("   pip install pandas")
    if "openpyxl" in missing: print("   pip install openpyxl")
    if "PyPDF2" in missing: print("   pip install PyPDF2")

# ======================================================================
# Parâmetros fixos do seu ambiente
# ======================================================================
BASE_PROJETOS = r"C:\Users\126815\OneDrive - paguemenos.com.br\Área de Trabalho\NFS\2025"

PROJETOS = [
    "ESPECIAL", "LAUDO DE RISCO", "LAUDO DE RISCO CD", "LAUDO DE RISCO MATRIZ",
    "LOJAS NOVAS", "MISSOES HIGIENICAS", "OUTROS", "PROJETO ESPECIAL",
    "RESIZE", "RETROFIT", "VIRADA DE BANDEIRA",
]

LOOKBACK_DAYS = 15
SUBPASTA_PATH = ""       # ex.: "Engenharia/NFs"
DIAGNOSTICO = True
LOG_IGNORADOS = False

# ======================================================================
# UI Terminal
# ======================================================================
def select_projeto_menu(opcoes: list) -> str:
    print("\n📁 Selecione o PROJETO:")
    for i, nome in enumerate(opcoes, start=1):
        print(f" {i}. {nome}")
    print(" 0. Cancelar")
    while True:
        try:
            op = int(input("Digite o número da opção: ").strip())
            if op == 0: return None
            if 1 <= op <= len(opcoes):
                projeto = opcoes[op - 1]
                print(f"✅ Selecionado: {projeto}")
                return projeto
        except ValueError:
            pass
        print("Opção inválida. Tente novamente.")

def ask_fornecedor_nome() -> str:
    nome = input("\nDigite o NOME exato do FORNECEDOR (ex.: LUMICENTER): ").strip()
    while not nome:
        nome = input("Digite o NOME do FORNECEDOR: ").strip()
    return nome

# ======================================================================
# Utils locais
# ======================================================================
def _ensure_dirs(path): os.makedirs(path, exist_ok=True)

def _sanitize_filename(name: str) -> str:
    name = (name or "").strip().replace("\n", " ").replace("\r", " ")
    name = re.sub(r'[<>:"/\\|?*]', "_", name)
    name = re.sub(r"\s{2,}", " ", name)
    return name

def _md5_file(path: str) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""): h.update(chunk)
    return h.hexdigest()

def _normalize_text(text: str) -> str:
    if not text: return ""
    t = re.sub(r"[\n\r\t]+", " ", text)
    t = re.sub(r"\s{2,}", " ", t)
    return t.upper()

# 🔧 Normalização robusta para PDF (remove acentos combinados, comprime espaços)
def _norm_pdf(text: str) -> str:
    if not text:
        return ""
    t = unicodedata.normalize("NFD", text)
    t = "".join(ch for ch in t if unicodedata.category(ch) != "Mn")
    t = re.sub(r"[\r\n\t]+", " ", t)
    t = re.sub(r"\s{2,}", " ", t)
    return t.upper().strip()

def _parse_brl_to_float(v):
    if v is None: return None
    s = str(v)
    s = re.sub(r"[Rr]\$\s*", "", s)
    s = s.replace(".", "TEMP").replace(",", ".").replace("TEMP", "")
    s = re.sub(r"[^\d\.]", "", s)
    try: return float(s)
    except: return None

def _to_naive_local(d: dt.datetime) -> dt.datetime or None:
    if d is None: return None
    try:
        if getattr(d, "tzinfo", None) is not None:
            return d.astimezone().replace(tzinfo=None)
        return d
    except Exception:
        try: return dt.datetime.fromtimestamp(d.timestamp()).replace(tzinfo=None)
        except Exception: return None

# 🔧 NOVO: remover zeros à esquerda de uma string numérica
def _strip_leading_zeros_num(s: str) -> str:
    s = (s or "").strip()
    s = re.sub(r"^0+", "", s)
    return s or "0"

# ======================================================================
# CNPJ & Loja
# ======================================================================
def _cnpj_digits(s: str) -> str:
    d = re.sub(r"\D", "", s or "")
    return d if len(d) == 14 else ""

def _cnpj_is_valid(d: str) -> bool:
    d = _cnpj_digits(d)
    if len(d) != 14 or len(set(d)) == 1:
        return False
    def dv_calc(digs, pesos):
        soma = sum(int(digs[i]) * pesos[i] for i in range(len(pesos)))
        r = soma % 11
        return '0' if r < 2 else str(11 - r)
    base = d[:12]
    dv1 = dv_calc(base, [5,4,3,2,9,8,7,6,5,4,3,2])
    dv2 = dv_calc(base + dv1, [6,5,4,3,2,9,8,7,6,5,4,3,2])
    return d[-2:] == dv1 + dv2

# 🔧 MANTIDO: sua regra de derivar loja pelo CNPJ
def _derive_loja_from_cnpj(cnpj_raw: str, origem_cnpj: str) -> tuple:
    d = _cnpj_digits(cnpj_raw)
    if not d: return None, origem_cnpj
    seq = d[8:12]
    origem_loja = origem_cnpj
    if d.startswith("04"):
        loja = "7" + seq[1:]; origem_loja += " + Regra IMIFARMA"
    elif d.startswith("06"):
        loja = seq.lstrip("0") or "0"; origem_loja += " + Regra Pague Menos"
    else:
        loja = seq.lstrip("0") or seq
    return loja, origem_loja

# ======================================================================
# Extratores auxiliares
# ======================================================================
def _find_oc_candidates(text_upper: str) -> list:
    text_upper = _norm_pdf(text_upper)   # melhora estabilidade dos regex
    oc_pattern = r"\b(45\d{8,10})\b"
    cands = []
    labels = ["ORDEM DE COMPRA", "OC", "PEDIDO", "PEDIDO CLIENTE", "INFORMACOES COMPLEMENTARES"]
    for lbl in labels:
        m_lbl = re.search(lbl, text_upper)
        if m_lbl:
            start = max(0, m_lbl.start() - 2000)
            bloco = text_upper[start: m_lbl.start()+4000]
            for m in re.finditer(oc_pattern, bloco):
                cands.append((start + m.start(), m.group(1), 5))
    for m in re.finditer(oc_pattern, text_upper):
        cands.append((m.start(), m.group(1), 1))
    seen, uniq = set(), []
    for pos, oc, score in cands:
        if oc not in seen:
            seen.add(oc); uniq.append((pos, oc, score))
    return uniq

# ======================================================================
# Extrator de dados do PDF (AJUSTADO)
# ======================================================================
def extrair_dados_pdf(caminho_pdf, debug=False):
    if PdfReader is None:
        return None, None, None, None, None, None, {}

    try:
        reader = PdfReader(caminho_pdf)
        texto_completo = "\n".join([p.extract_text() or "" for p in reader.pages])

        # Normalização para regex robusto (acentos e quebras)
        T = _norm_pdf(texto_completo)

        # ---------------------------------------------------------------
        # 1) CNPJ do TOMADOR (não usar o do prestador)
        # ---------------------------------------------------------------
        cnpj_tomador = None

        m_tom = re.search(r"\bTOMADOR\b[^0-9]{0,300}?(\d{2}\.\d{3}\.\d{3}/\d{4}-\d{2})", T)
        if m_tom:
            cnpj_tomador = _cnpj_digits(m_tom.group(1))
        else:
            # fallback: segundo CNPJ do documento
            all_cnpjs = re.findall(r"\d{2}\.\d{3}\.\d{3}/\d{4}-\d{2}", T)
            if len(all_cnpjs) >= 2:
                cnpj_tomador = _cnpj_digits(all_cnpjs[1])
            elif all_cnpjs:
                cnpj_tomador = _cnpj_digits(all_cnpjs[0])

        # ---------------------------------------------------------------
        # 2) Loja — mantém exatamente sua regra: derivar pelo CNPJ
        # ---------------------------------------------------------------
        loja, origem_loja = _derive_loja_from_cnpj(cnpj_tomador, "CNPJ TOMADOR")

        # ---------------------------------------------------------------
        # 3) Número da NF (lida com acentos/quebras e rótulos alternativos)
        # ---------------------------------------------------------------
        nf = None

        # "NUMERO DA NOTA" (pode estar em linha seguinte)
        m_nf = re.search(r"\bNUMERO\s+DA\s+NOTA\b[^\d]{0,30}(\d{4,})", T)
        if m_nf:
            nf = m_nf.group(1)

        if not nf:
            padroes = [
                r"\bNFS?-?E\b[^\d]{0,15}N[ºO]?\s*[:\-]?\s*(\d{4,})",
                r"\bNF-?E\b[^\d]{0,15}N[ºO]?\s*[:\-]?\s*(\d{4,})",
            ]
            for p in padroes:
                m = re.search(p, T)
                if m:
                    nf = m.group(1)
                    break

        # Evita captura errada como telefone (ex.: (98) ....)
        if nf:
            span = re.search(nf, T)
            if span:
                ctx = T[max(0, span.start()-10): span.end()+10]
                if re.search(r"\(\d{2}\)", ctx) or "TELEFONE" in ctx:
                    nf = None

        # 🔧 NOVO: remove zeros à esquerda da NF aqui (padrão global)
        if nf:
            nf = _strip_leading_zeros_num(nf)

        # ---------------------------------------------------------------
        # 4) Demais campos (mantém sua lógica)
        # ---------------------------------------------------------------
        serie = (re.search(r"\bSERIE\b[^\dA-Z]{0,10}(\d+|[A-Z]+)\b", T) or [None, None])[1]
        valor = (re.search(r"VALOR\s*TOTAL[^\d]{0,20}([\d\.,]{4,})", T) or [None, None])[1]
        chave = (re.search(r"\bCHAVE\b[^\d]{0,20}(\d{44})", T) or [None, None])[1]

        oc = None
        oc_cands = _find_oc_candidates(T)
        if oc_cands:
            oc = max(oc_cands, key=lambda x: x[2])[1]

        data_emissao = (
            re.search(r"(?:DATA(?:\s*E\s*HORA)?\s*DE?\s*EMISSAO)[^\d]{0,60}(\d{2}/\d{2}/\d{4})", T)
            or [None, None]
        )[1]

        extras = {
            "Valor Total": valor,
            "ordem_compra": oc,
            "pedido": oc,
            "data_emissao": data_emissao,
            "destinatario_cnpj": cnpj_tomador
        }

        if debug:
            print(f"[DEBUG] NF={nf} | LOJA={loja} | ORIGEM={origem_loja} | CNPJ_TOMADOR={cnpj_tomador}")

        return nf, loja, "PDF", origem_loja, serie, chave, extras

    except Exception as e:
        if debug:
            print(f"Erro: {e}")
        return None, None, None, None, None, None, {}

# ======================================================================
# Outlook (Win32)
# ======================================================================
PR_SMTP_ADDRESS = "http://schemas.microsoft.com/mapi/proptag/0x39FE001E"
SUBJECT_PATTERNS = [r"\bnota\s*fiscal\b", r"\bnf-?e\b", r"\bdanfe\b", r"\bnf\b", r"\bnfs-?e\b"]

def subject_is_nf(subject: str) -> bool:
    s = subject or ""
    return any(re.search(p, s, re.IGNORECASE) for p in SUBJECT_PATTERNS)

def resolve_sender_smtp(mail_item):
    try:
        sender = mail_item.Sender
        if sender:
            return (sender.AddressEntry.PropertyAccessor.GetProperty(PR_SMTP_ADDRESS) or "").lower()
    except:
        pass
    return (getattr(mail_item, "SenderEmailAddress", "") or "").lower()

def resolve_subfolder(root_folder, path_str):
    if not path_str: return root_folder
    cur = root_folder
    for part in path_str.split("/"):
        if part: cur = cur.Folders(part)
    return cur

def coletar_anexos_outlook_multi(filter_senders_lower):
    if win32 is None: return []
    outlook = win32.Dispatch("Outlook.Application").GetNamespace("MAPI")
    dt_from = (dt.datetime.now() - dt.timedelta(days=LOOKBACK_DAYS))

    coletados = []
    try:
        inbox = outlook.GetDefaultFolder(6)
        folder = resolve_subfolder(inbox, SUBPASTA_PATH.replace("\\", "/"))
        items = folder.Items
        items.Sort("[ReceivedTime]", True)
        it = items.GetFirst()
        while it:
            try:
                received = _to_naive_local(getattr(it, "ReceivedTime", None))
                if received and received < dt_from:
                    break

                sender_addr = resolve_sender_smtp(it)
                subject = getattr(it, "Subject", "")

                # Filtro de remetente ou assunto
                match = not filter_senders_lower or any(s in sender_addr for s in filter_senders_lower)
                if match or subject_is_nf(subject):
                    temp_dir = os.path.join(os.getcwd(), "_temp_nfs"); _ensure_dirs(temp_dir)
                    for att in it.Attachments:
                        if str(att.FileName).lower().endswith(".pdf"):
                            temp_path = os.path.join(temp_dir, _sanitize_filename(att.FileName))
                            att.SaveAsFile(temp_path)
                            coletados.append({
                                "sender_email": sender_addr,
                                "subject": subject,
                                "received": received,
                                "temp_path": temp_path,
                                "orig_filename": att.FileName
                            })
            except:
                pass
            it = items.GetNext()
    except Exception as e:
        print(f"Erro Outlook: {e}")
    return coletados

# ======================================================================
# MAIN
# ======================================================================
def main():
    projeto = select_projeto_menu(PROJETOS)
    if not projeto: return
    fornecedor = ask_fornecedor_nome()

    pasta_projeto = os.path.join(BASE_PROJETOS, projeto)
    pasta_fornecedor = os.path.join(pasta_projeto, fornecedor)

    if not os.path.isdir(pasta_projeto):
        print(f"❌ Pasta do projeto não existe: {pasta_projeto}")
        return

    raw_emails = input("\nFiltro de e-mails (vazio para todos): ").strip()
    FILTER_SENDERS = [e.strip().lower() for e in raw_emails.split(",") if e.strip()]

    registros = coletar_anexos_outlook_multi(FILTER_SENDERS)
    print(f"📥 PDFs encontrados: {len(registros)}")

    base_registros = []
    for reg in registros:
        nf, loja, _, origem_lo, serie, chave, extras = extrair_dados_pdf(reg["temp_path"])

        # Nome do arquivo — mantém seu padrão
        novo_nome = f"NF {nf} - LJ {loja}.pdf" if nf and loja else f"NF {nf or 'SEM_NUM'}.pdf"
        destino = os.path.join(pasta_fornecedor, novo_nome)

        status = "OK"
        if os.path.isdir(pasta_fornecedor):
            try:
                if os.path.exists(destino):
                    destino = os.path.join(
                        pasta_fornecedor,
                        f"{os.path.splitext(novo_nome)[0]}_{_md5_file(reg['temp_path'])[:5]}.pdf"
                    )
                shutil.move(reg["temp_path"], destino)
            except Exception as e:
                status = f"Erro: {e}"
        else:
            status = "Pasta Fornecedor Ausente"

        base_registros.append({
            "Projeto": projeto,
            "Fornecedor": fornecedor,
            "NF": nf,
            "Loja": loja,
            "Valor Total": extras.get("Valor Total"),
            "Pedido": extras.get("pedido"),
            "Recebido Em": reg["received"],
            "Status": status,
            "Arquivo": os.path.basename(destino)
        })

    if base_registros and pd:
        df = pd.DataFrame(base_registros)
        hoje = dt.datetime.now().strftime("%Y-%m-%d")
        excel_path = os.path.join(pasta_projeto, f"BASE_NFs_{fornecedor}_{hoje}.xlsx")
        df.to_excel(excel_path, index=False)
        print(f"✅ Excel gerado: {excel_path}")

if __name__ == "__main__":
    main()

In [2]:
!pip install pypdf2

Defaulting to user installation because normal site-packages is not writeable


In [1]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

import os, re, sys, csv, hashlib, shutil, datetime as dt
import unicodedata

# --------- Dependências críticas ---------
missing = []
try:
    import win32com.client as win32
except Exception:
    win32 = None
    missing.append("pywin32")
try:
    import pandas as pd
except Exception:
    pd = None
    missing.append("pandas")
try:
    from PyPDF2 import PdfReader
except Exception:
    PdfReader = None
    missing.append("PyPDF2")
try:
    import openpyxl
except Exception:
    missing.append("openpyxl")

if missing:
    print("⚠️ Dependências ausentes:", ", ".join(missing))
    print("   Instale no Anaconda Prompt:")
    if "pywin32" in missing: print("   pip install pywin32")
    if "pandas" in missing: print("   pip install pandas")
    if "openpyxl" in missing: print("   pip install openpyxl")
    if "PyPDF2" in missing: print("   pip install PyPDF2")

# ======================================================================
# Parâmetros fixos do seu ambiente
# ======================================================================
BASE_PROJETOS = r"C:\Users\126815\OneDrive - paguemenos.com.br\Área de Trabalho\NFS\2025"

PROJETOS = [
    "ESPECIAL", "LAUDO DE RISCO", "LAUDO DE RISCO CD", "LAUDO DE RISCO MATRIZ",
    "LOJAS NOVAS", "MISSOES HIGIENICAS", "OUTROS", "PROJETO ESPECIAL",
    "RESIZE", "RETROFIT", "VIRADA DE BANDEIRA",
]

LOOKBACK_DAYS = 15
SUBPASTA_PATH = ""       # ex.: "Engenharia/NFs"
DIAGNOSTICO = True
LOG_IGNORADOS = False

# ======================================================================
# UI Terminal
# ======================================================================
def select_projeto_menu(opcoes: list) -> str:
    print("\n📁 Selecione o PROJETO:")
    for i, nome in enumerate(opcoes, start=1):
        print(f" {i}. {nome}")
    print(" 0. Cancelar")
    while True:
        try:
            op = int(input("Digite o número da opção: ").strip())
            if op == 0: return None
            if 1 <= op <= len(opcoes):
                projeto = opcoes[op - 1]
                print(f"✅ Selecionado: {projeto}")
                return projeto
        except ValueError:
            pass
        print("Opção inválida. Tente novamente.")

def ask_fornecedor_nome() -> str:
    nome = input("\nDigite o NOME exato do FORNECEDOR (ex.: LUMICENTER): ").strip()
    while not nome:
        nome = input("Digite o NOME do FORNECEDOR: ").strip()
    return nome

# ======================================================================
# Utils locais
# ======================================================================
def _ensure_dirs(path): os.makedirs(path, exist_ok=True)

def _sanitize_filename(name: str) -> str:
    name = (name or "").strip().replace("\n", " ").replace("\r", " ")
    name = re.sub(r'[<>:"/\\|?*]', "_", name)
    name = re.sub(r"\s{2,}", " ", name)
    return name

def _md5_file(path: str) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""): h.update(chunk)
    return h.hexdigest()

def _normalize_text(text: str) -> str:
    if not text: return ""
    t = re.sub(r"[\n\r\t]+", " ", text)
    t = re.sub(r"\s{2,}", " ", t)
    return t.upper()

# 🔧 Normalização robusta para PDF (remove acentos combinados, comprime espaços)
def _norm_pdf(text: str) -> str:
    if not text:
        return ""
    t = unicodedata.normalize("NFD", text)
    t = "".join(ch for ch in t if unicodedata.category(ch) != "Mn")
    t = re.sub(r"[\r\n\t]+", " ", t)
    t = re.sub(r"\s{2,}", " ", t)
    return t.upper().strip()

def _parse_brl_to_float(v):
    if v is None: return None
    s = str(v)
    s = re.sub(r"[Rr]\$\s*", "", s)
    s = s.replace(".", "TEMP").replace(",", ".").replace("TEMP", "")
    s = re.sub(r"[^\d\.]", "", s)
    try: return float(s)
    except: return None

def _to_naive_local(d: dt.datetime) -> dt.datetime or None:
    if d is None: return None
    try:
        if getattr(d, "tzinfo", None) is not None:
            return d.astimezone().replace(tzinfo=None)
        return d
    except Exception:
        try: return dt.datetime.fromtimestamp(d.timestamp()).replace(tzinfo=None)
        except Exception: return None

# 🔧 NOVO: remover zeros à esquerda de uma string numérica
def _strip_leading_zeros_num(s: str) -> str:
    s = (s or "").strip()
    s = re.sub(r"^0+", "", s)
    return s or "0"

# ======================================================================
# CNPJ & Loja
# ======================================================================
def _cnpj_digits(s: str) -> str:
    d = re.sub(r"\D", "", s or "")
    return d if len(d) == 14 else ""

def _cnpj_is_valid(d: str) -> bool:
    d = _cnpj_digits(d)
    if len(d) != 14 or len(set(d)) == 1:
        return False
    def dv_calc(digs, pesos):
        soma = sum(int(digs[i]) * pesos[i] for i in range(len(pesos)))
        r = soma % 11
        return '0' if r < 2 else str(11 - r)
    base = d[:12]
    dv1 = dv_calc(base, [5,4,3,2,9,8,7,6,5,4,3,2])
    dv2 = dv_calc(base + dv1, [6,5,4,3,2,9,8,7,6,5,4,3,2])
    return d[-2:] == dv1 + dv2

# 🔧 MANTIDO: sua regra de derivar loja pelo CNPJ
def _derive_loja_from_cnpj(cnpj_raw: str, origem_cnpj: str) -> tuple:
    d = _cnpj_digits(cnpj_raw)
    if not d: return None, origem_cnpj
    seq = d[8:12]
    origem_loja = origem_cnpj
    if d.startswith("04"):
        loja = "7" + seq[1:]; origem_loja += " + Regra IMIFARMA"
    elif d.startswith("06"):
        loja = seq.lstrip("0") or "0"; origem_loja += " + Regra Pague Menos"
    else:
        loja = seq.lstrip("0") or seq
    return loja, origem_loja

# ======================================================================
# Extratores auxiliares
# ======================================================================
def _find_oc_candidates(text_upper: str) -> list:
    text_upper = _norm_pdf(text_upper)   # melhora estabilidade dos regex
    oc_pattern = r"\b(45\d{8,10})\b"
    cands = []
    labels = ["ORDEM DE COMPRA", "OC", "PEDIDO", "PEDIDO CLIENTE", "INFORMACOES COMPLEMENTARES"]
    for lbl in labels:
        m_lbl = re.search(lbl, text_upper)
        if m_lbl:
            start = max(0, m_lbl.start() - 2000)
            bloco = text_upper[start: m_lbl.start()+4000]
            for m in re.finditer(oc_pattern, bloco):
                cands.append((start + m.start(), m.group(1), 5))
    for m in re.finditer(oc_pattern, text_upper):
        cands.append((m.start(), m.group(1), 1))
    seen, uniq = set(), []
    for pos, oc, score in cands:
        if oc not in seen:
            seen.add(oc); uniq.append((pos, oc, score))
    return uniq

# ======================================================================
# Extrator de dados do PDF (AJUSTADO)
# ======================================================================
def extrair_dados_pdf(caminho_pdf, debug=False):
    if PdfReader is None:
        return None, None, None, None, None, None, {}

    try:
        reader = PdfReader(caminho_pdf)
        texto_completo = "\n".join([p.extract_text() or "" for p in reader.pages])

        # Normalização para regex robusto (acentos e quebras)
        T = _norm_pdf(texto_completo)

        # ---------------------------------------------------------------
        # 1) CNPJ do TOMADOR (não usar o do prestador)
        # ---------------------------------------------------------------
        cnpj_tomador = None

        m_tom = re.search(r"\bTOMADOR\b[^0-9]{0,300}?(\d{2}\.\d{3}\.\d{3}/\d{4}-\d{2})", T)
        if m_tom:
            cnpj_tomador = _cnpj_digits(m_tom.group(1))
        else:
            # fallback: segundo CNPJ do documento
            all_cnpjs = re.findall(r"\d{2}\.\d{3}\.\d{3}/\d{4}-\d{2}", T)
            if len(all_cnpjs) >= 2:
                cnpj_tomador = _cnpj_digits(all_cnpjs[1])
            elif all_cnpjs:
                cnpj_tomador = _cnpj_digits(all_cnpjs[0])

        # ---------------------------------------------------------------
        # 2) Loja — mantém exatamente sua regra: derivar pelo CNPJ
        # ---------------------------------------------------------------
        loja, origem_loja = _derive_loja_from_cnpj(cnpj_tomador, "CNPJ TOMADOR")

        # ---------------------------------------------------------------
        # 3) Número da NF (lida com acentos/quebras e rótulos alternativos)
        # ---------------------------------------------------------------
        nf = None

        # "NUMERO DA NOTA" (pode estar em linha seguinte)
        m_nf = re.search(r"\bNUMERO\s+DA\s+NOTA\b[^\d]{0,30}(\d{4,})", T)
        if m_nf:
            nf = m_nf.group(1)

        if not nf:
            padroes = [
                r"\bNFS?-?E\b[^\d]{0,15}N[ºO]?\s*[:\-]?\s*(\d{4,})",
                r"\bNF-?E\b[^\d]{0,15}N[ºO]?\s*[:\-]?\s*(\d{4,})",
            ]
            for p in padroes:
                m = re.search(p, T)
                if m:
                    nf = m.group(1)
                    break

        # Evita captura errada como telefone (ex.: (98) ....)
        if nf:
            span = re.search(nf, T)
            if span:
                ctx = T[max(0, span.start()-10): span.end()+10]
                if re.search(r"\(\d{2}\)", ctx) or "TELEFONE" in ctx:
                    nf = None

        # 🔧 NOVO: remove zeros à esquerda da NF aqui (padrão global)
        if nf:
            nf = _strip_leading_zeros_num(nf)

        # ---------------------------------------------------------------
        # 4) Demais campos (mantém sua lógica)
        # ---------------------------------------------------------------
        serie = (re.search(r"\bSERIE\b[^\dA-Z]{0,10}(\d+|[A-Z]+)\b", T) or [None, None])[1]
        valor = (re.search(r"VALOR\s*TOTAL[^\d]{0,20}([\d\.,]{4,})", T) or [None, None])[1]
        chave = (re.search(r"\bCHAVE\b[^\d]{0,20}(\d{44})", T) or [None, None])[1]

        oc = None
        oc_cands = _find_oc_candidates(T)
        if oc_cands:
            oc = max(oc_cands, key=lambda x: x[2])[1]

        data_emissao = (
            re.search(r"(?:DATA(?:\s*E\s*HORA)?\s*DE?\s*EMISSAO)[^\d]{0,60}(\d{2}/\d{2}/\d{4})", T)
            or [None, None]
        )[1]

        extras = {
            "Valor Total": valor,
            "ordem_compra": oc,
            "pedido": oc,
            "data_emissao": data_emissao,
            "destinatario_cnpj": cnpj_tomador
        }

        if debug:
            print(f"[DEBUG] NF={nf} | LOJA={loja} | ORIGEM={origem_loja} | CNPJ_TOMADOR={cnpj_tomador}")

        return nf, loja, "PDF", origem_loja, serie, chave, extras

    except Exception as e:
        if debug:
            print(f"Erro: {e}")
        return None, None, None, None, None, None, {}

# ======================================================================
# Outlook (Win32)
# ======================================================================
PR_SMTP_ADDRESS = "http://schemas.microsoft.com/mapi/proptag/0x39FE001E"
SUBJECT_PATTERNS = [r"\bnota\s*fiscal\b", r"\bnf-?e\b", r"\bdanfe\b", r"\bnf\b", r"\bnfs-?e\b"]

def subject_is_nf(subject: str) -> bool:
    s = subject or ""
    return any(re.search(p, s, re.IGNORECASE) for p in SUBJECT_PATTERNS)

def resolve_sender_smtp(mail_item):
    try:
        sender = mail_item.Sender
        if sender:
            return (sender.AddressEntry.PropertyAccessor.GetProperty(PR_SMTP_ADDRESS) or "").lower()
    except:
        pass
    return (getattr(mail_item, "SenderEmailAddress", "") or "").lower()

def resolve_subfolder(root_folder, path_str):
    if not path_str: return root_folder
    cur = root_folder
    for part in path_str.split("/"):
        if part: cur = cur.Folders(part)
    return cur

def coletar_anexos_outlook_multi(filter_senders_lower):
    if win32 is None: return []
    outlook = win32.Dispatch("Outlook.Application").GetNamespace("MAPI")
    dt_from = (dt.datetime.now() - dt.timedelta(days=LOOKBACK_DAYS))

    coletados = []
    try:
        inbox = outlook.GetDefaultFolder(6)
        folder = resolve_subfolder(inbox, SUBPASTA_PATH.replace("\\", "/"))
        items = folder.Items
        items.Sort("[ReceivedTime]", True)
        it = items.GetFirst()
        while it:
            try:
                received = _to_naive_local(getattr(it, "ReceivedTime", None))
                if received and received < dt_from:
                    break

                sender_addr = resolve_sender_smtp(it)
                subject = getattr(it, "Subject", "")

                # Filtro de remetente ou assunto
                match = not filter_senders_lower or any(s in sender_addr for s in filter_senders_lower)
                if match or subject_is_nf(subject):
                    temp_dir = os.path.join(os.getcwd(), "_temp_nfs"); _ensure_dirs(temp_dir)
                    for att in it.Attachments:
                        if str(att.FileName).lower().endswith(".pdf"):
                            temp_path = os.path.join(temp_dir, _sanitize_filename(att.FileName))
                            att.SaveAsFile(temp_path)
                            coletados.append({
                                "sender_email": sender_addr,
                                "subject": subject,
                                "received": received,
                                "temp_path": temp_path,
                                "orig_filename": att.FileName
                            })
            except:
                pass
            it = items.GetNext()
    except Exception as e:
        print(f"Erro Outlook: {e}")
    return coletados

# ======================================================================
# MAIN
# ======================================================================
def main():
    projeto = select_projeto_menu(PROJETOS)
    if not projeto: return
    fornecedor = ask_fornecedor_nome()

    pasta_projeto = os.path.join(BASE_PROJETOS, projeto)
    pasta_fornecedor = os.path.join(pasta_projeto, fornecedor)

    if not os.path.isdir(pasta_projeto):
        print(f"❌ Pasta do projeto não existe: {pasta_projeto}")
        return

    raw_emails = input("\nFiltro de e-mails (vazio para todos): ").strip()
    FILTER_SENDERS = [e.strip().lower() for e in raw_emails.split(",") if e.strip()]

    registros = coletar_anexos_outlook_multi(FILTER_SENDERS)
    print(f"📥 PDFs encontrados: {len(registros)}")

    base_registros = []
    for reg in registros:
        nf, loja, _, origem_lo, serie, chave, extras = extrair_dados_pdf(reg["temp_path"])

        # Nome do arquivo — mantém seu padrão
        novo_nome = f"NF {nf} - LJ {loja}.pdf" if nf and loja else f"NF {nf or 'SEM_NUM'}.pdf"
        destino = os.path.join(pasta_fornecedor, novo_nome)

        status = "OK"
        if os.path.isdir(pasta_fornecedor):
            try:
                if os.path.exists(destino):
                    destino = os.path.join(
                        pasta_fornecedor,
                        f"{os.path.splitext(novo_nome)[0]}_{_md5_file(reg['temp_path'])[:5]}.pdf"
                    )
                shutil.move(reg["temp_path"], destino)
            except Exception as e:
                status = f"Erro: {e}"
        else:
            status = "Pasta Fornecedor Ausente"

        base_registros.append({
            "Projeto": projeto,
            "Fornecedor": fornecedor,
            "NF": nf,
            "Loja": loja,
            "Valor Total": extras.get("Valor Total"),
            "Pedido": extras.get("pedido"),
            "Recebido Em": reg["received"],
            "Status": status,
            "Arquivo": os.path.basename(destino)
        })

    if base_registros and pd:
        df = pd.DataFrame(base_registros)
        hoje = dt.datetime.now().strftime("%Y-%m-%d")
        excel_path = os.path.join(pasta_projeto, f"BASE_NFs_{fornecedor}_{hoje}.xlsx")
        df.to_excel(excel_path, index=False)
        print(f"✅ Excel gerado: {excel_path}")

if __name__ == "__main__":
    main()


📁 Selecione o PROJETO:
 1. ESPECIAL
 2. LAUDO DE RISCO
 3. LAUDO DE RISCO CD
 4. LAUDO DE RISCO MATRIZ
 5. LOJAS NOVAS
 6. MISSOES HIGIENICAS
 7. OUTROS
 8. PROJETO ESPECIAL
 9. RESIZE
 10. RETROFIT
 11. VIRADA DE BANDEIRA
 0. Cancelar


Digite o número da opção:  10


✅ Selecionado: RETROFIT



Digite o NOME exato do FORNECEDOR (ex.: LUMICENTER):  PLANAC

Filtro de e-mails (vazio para todos):  Polyana Santos <financeiro@planac.eng.br>


📥 PDFs encontrados: 2
✅ Excel gerado: C:\Users\126815\OneDrive - paguemenos.com.br\Área de Trabalho\NFS\2025\RETROFIT\BASE_NFs_PLANAC_2026-05-12.xlsx
